# 17. Investment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/macro102-site/blob/main/notebooks/ch17-investment.ipynb)

## Learning goals

Investment links the present to the future: a firm that buys a machine today gives up money now
for output later. One tool, the **arbitrage equation**, explains investment in machines, in stocks
and in houses. By the end of this chapter you will be able to:

- use an arbitrage equation to compare two ways of investing the same money, and derive from it the
  **user cost of capital**, with and without a **corporate income tax**;
- derive the **desired capital stock**, where the marginal product of capital equals the user cost, and
  the **investment rate** it implies, step by step from a Cobb-Douglas production function;
- read the **comparative statics** of desired capital and the investment rate as elasticities, and rebuild
  Table 17.1 of corporate taxes and user costs for any year;
- derive the price of a stock as the present value of its dividends, $\text{dividend}/(R-g)$, and use it
  to read the **price-earnings ratio** of the stock market, **Tobin's $q$** and the price of a house;
- describe the components of private investment, in the U.S. and in 34 other countries, and how they move in
  recessions;
- write Python functions with default values, draw Plotly figures with sliders and buttons, reshape
  tables with `rename`, `melt` and `merge`, and repeat steps with `for` and `while` loops.

## Setup

We first load the Python libraries we need. You only have to run this cell once.

> **Python note.** `import` loads a *library*, a toolbox written by other people. We give each
> library a short nickname, for example `np` for NumPy, so that the code stays short. Text after
> a `#` is a *comment*: Python ignores it, and it is there for human readers. The `try ... except`
> block runs one Colab-only line and simply skips it anywhere else.

In [1]:
# Libraries used in this chapter
import numpy as np                              # numbers and arrays
import pandas as pd                             # data tables
import plotly.express as px                     # quick charts from data tables
import plotly.graph_objects as go               # charts built piece by piece (model figures)
import plotly.io as pio
from ipywidgets import interact, interact_manual, FloatSlider, ToggleButtons, Dropdown, fixed

# One light blue/green look for every chart
BLUE, TEAL, GREEN, GRAY = "#16528c", "#008070", "#7fb23a", "#8a96a3"
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.update(width=860, height=480)   # same size in Colab and on the website
px.defaults.color_discrete_sequence = [BLUE, TEAL, GREEN, "#5aa0d8", "#0b4f47"]
WIDE = {"description_width": "initial"}         # show the full slider labels

# Colab only: let Plotly charts appear inside the interactive widgets
try:
    import google.colab
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass                                        # not running in Colab: nothing to do

# The data files of this chapter live in the book's GitHub repository
DATA_URL = "https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/data/ch17/"

The next cell defines a helper that only the book website needs. Run it and move on.

In [2]:
# @title Helper for the website versions (run this cell; you do not need to read it)
def slider_figure(make_figure, name, values, start, label=None):
    """Precompute make_figure(name=value) for every value and join the results with one slider.

    The website cannot run Python, so each interactive figure also gets this precomputed version.
    """
    figures = [make_figure(**{name: v}) for v in values]
    first = values.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which precomputed figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    steps = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(),
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        visible = [o == i for o in owner]
        steps.append(dict(method="update", label=str(values[i]), args=[{"visible": visible}, layout]))
    fig.update_layout(sliders=[dict(active=first, steps=steps, pad=dict(t=60),
                                    currentvalue=dict(prefix=(label or name) + " = "))])
    return fig

## Model overview

Investment is the part of GDP that builds the future: new machines, buildings, houses and inventories. This chapter asks how much capital a firm wants to own, and how that choice turns into a flow of investment. The answer rests on one simple idea, the **arbitrage equation**: when two safe ways of using the same money are both in use, they must pay the same return. If one paid more, moving a dollar into it would raise profits, so an investor who already maximizes profits cannot see such a gap.

We apply that idea five times. First to a firm that compares a bank deposit with a new machine, which gives the **user cost of capital**. Then we add a corporate income tax. With a Cobb-Douglas production function the condition pins down the **desired capital stock** and the economy's **investment rate**. Next the same comparison prices a share of stock and explains the **price-earnings ratio**, and it links the stock market to investment through **Tobin's $q$**. Last, it prices a house bought with a mortgage. The block `### The model in code` then turns the results into Python functions and checks each of them with numbers.

A bar over a letter marks a **parameter**, a number taken from outside the model, such as the depreciation rate $\bar{d}$ or productivity $\bar{A}$. The interest rate $R$ is a real rate, the cost of funds for one year, and $\Delta x \equiv x_{t+1} - x_t$ is the change of any variable over one year. All the rates are written as decimals: $R = 0.02$ means 2 percent a year.

### Investment and the user cost of capital

#### Assumptions

1. **Two uses of cash.** A firm has some money to spare. It can deposit it in a bank, where it earns the interest rate $R$ for one year. Or it can buy one unit of capital (a pizza oven, say) at the price $p_k$, use it for a year in production, and sell it at the end of the year at the new price $p_{k,t+1} = p_k + \Delta p_k$.
2. **No risk.** The firm knows the return of both options in advance, so it compares them only by their size.
3. **Diminishing returns.** One more unit of capital adds the **marginal product of capital**, $MPK$, to output. $MPK$ falls as the firm owns more capital, holding its other inputs fixed (Chapter 4).
4. **Wear and tear.** A fraction $\bar{d}$ of capital wears out during the year, the depreciation rate of the Solow model ([Chapter 5](https://quarcs-lab.github.io/macro102-site/notebooks/ch05-the-solow-growth-model.html)).
5. **Profit maximization.** The firm moves money between the two uses until no switch can raise its profits.

In the simple production model of [Chapter 4](https://quarcs-lab.github.io/macro102-site/notebooks/ch04-a-model-of-production.html) the answer was to invest until $MPK = R$: the last unit of capital must earn the cost of the funds tied up in it. The arbitrage argument below generalizes that rule.

#### Solving the model

**Step 1: the arbitrage equation.** Put the same $p_k$ dollars into each option for one year and write what each pays. The bank pays interest on the sum. The machine pays its marginal product plus the change in its price, a **capital gain** if the price rises and a **capital loss** if it falls. At the profit-maximizing choice the two must be equal:

$$\underbrace{R\cdot p_k}_{\text{deposit } p_k \text{ in the bank}} = \underbrace{MPK + \Delta p_k}_{\text{buy the machine, sell it after a year}}. \tag{1}$$

*Why the two sides must be equal.* If the machine paid more, the firm would buy one more machine. Because of diminishing returns, each extra machine lowers $MPK$, so buying continues until the right side has fallen to the left side. If the machine paid less, the firm would own fewer machines, which raises $MPK$. Either way the firm ends up where equation (1) holds.

**Step 2: returns per dollar.** Measure capital in units that cost one dollar today, $p_k = 1$, and divide both sides of (1) by $p_k$. The left side becomes $R$, and $\Delta p_k/p_k$ is the capital gain in percent. Solving for $MPK$:

$$R = MPK + \frac{\Delta p_k}{p_k} \quad\Longrightarrow\quad MPK = R - \frac{\Delta p_k}{p_k}.$$

With a constant price, $\Delta p_k = 0$, this is the rule of Chapter 4, $MPK = R$.

**Step 3: depreciation.** Wear and tear is a capital loss of its own: after a year of use, only $1-\bar{d}$ of the machine is left to sell. We therefore read $p_k$ as the price of a *new* unit and write depreciation as a separate term. The machine's return per dollar is now $MPK - \bar{d} + \Delta p_k/p_k$. Setting it equal to $R$ and moving $\bar{d}$ and the capital gain to the left:

$$MPK = \underbrace{R + \bar{d} - \frac{\Delta p_k}{p_k}}_{\text{user cost of capital, } uc}. \tag{2}$$

The **user cost of capital** $uc$ is what it costs to use one more unit of capital for one year. It has three parts: the interest the firm gives up by not keeping the money in the bank ($R$), the part of the machine that wears out ($\bar{d}$), and minus any rise in the machine's price, which the firm gets back when it sells.

With the values used throughout this notebook, $R = 0.02$, $\bar{d} = 0.08$ and no change in the price of capital, the user cost is $0.02 + 0.08 - 0 = 0.10$: each unit of capital must produce at least 10 cents of output a year to be worth owning.

#### What the model predicts

The firm invests until its $MPK$ curve, which falls with capital, meets the flat user-cost line (Fig. 1). Anything that raises the user cost shifts that line up and moves the desired capital stock to the left:

| A rise in ... | changes the user cost by | Desired capital | Which line moves |
|---|---|---|---|
| interest rate $R$ | $+1$ point per point | falls | user-cost line up |
| depreciation rate $\bar{d}$ | $+1$ point per point | falls | user-cost line up |
| expected capital gain $\Delta p_k/p_k$ | $-1$ point per point | rises | user-cost line down |
| productivity $\bar{A}$ | no change | rises | the $MPK$ curve shifts up |

Capital goods whose price keeps falling, such as computers, carry a capital loss every year, so their user cost is high. Buildings, whose land tends to become more valuable, often carry a capital gain, so their user cost is lower. A firm should therefore hold fewer units of the first kind and more of the second, for a given marginal product curve.

### Taxes and the user cost

#### Assumptions

Keep everything above and add a **corporate income tax**. The government takes a share $\tau$ of the extra income that a unit of capital produces, so the firm keeps $(1-\tau)MPK$. As in the textbook's version, depreciation and capital gains are left untaxed. The price of a new unit of capital is still normalized to $p_k = 1$.

#### Solving the model

**Step 1: the arbitrage equation after tax.** The cost of the funds is still $R$. The return from a dollar of capital is the after-tax marginal product, minus depreciation, plus the capital gain:

$$\underbrace{R}_{\text{cost of the funds}} = \underbrace{(1-\tau)MPK - \bar{d} + \frac{\Delta p_k}{p_k}}_{\text{return on a dollar of capital}}.$$

**Step 2: solve for the marginal product.** Add $\bar{d}$ to both sides and subtract the capital gain, so that only the tax term stays on the right: $R + \bar{d} - \Delta p_k/p_k = (1-\tau)MPK$. Then divide both sides by $1-\tau$:

$$MPK = \underbrace{\frac{R + \bar{d} - \Delta p_k/p_k}{1-\tau}}_{\text{user cost with the tax}}. \tag{3}$$

The tax divides the user cost by $1-\tau$. With our baseline, a tax of $\tau = 0.35$ raises it from $0.10$ to $0.10/0.65 \approx 0.154$, or about 15.4 percent.

*Intuition.* The government takes a slice of every unit's output but shares none of its costs: the interest, the wear and tear and the price change are the same as before. So a unit of capital must produce more before tax, by the factor $1/(1-\tau)$, to leave the firm the same return after tax.

#### What the model predicts

- **Sign.** A higher tax rate always raises the user cost, and so lowers the desired capital stock: the user-cost line of Fig. 2 shifts up and the firm moves up its $MPK$ curve.
- **Size.** Take logs of equation (3) ([Appendix C.4](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.4-logarithms)): $\ln uc = \ln(R + \bar{d} - \Delta p_k/p_k) - \ln(1-\tau)$. A 1 percent fall in the after-tax share $1-\tau$ raises the user cost by 1 percent.
- **The tax magnifies everything else.** Each point added to $R$ or $\bar{d}$ now raises the user cost by $1/(1-\tau)$ points, 1.54 points with $\tau = 0.35$.
- **Comparing countries.** If the user cost before tax is the same everywhere, equation (3) gives each country's user cost from one known country: $uc_i = uc_{\text{US}}\times(1-\tau_{\text{US}})/(1-\tau_i)$. With $uc_{\text{US}} = 0.10$ and $\tau_{\text{US}} = 0.21$, the user cost before tax is $0.10\times0.79 = 0.079$, which is how we rebuild Table 17.1 after Fig. 2.

### From desired capital to investment

#### Assumptions

1. **Production.** The firm (or the whole economy) produces with the Cobb-Douglas function of Chapter 4, $Y = \bar{A}K^{1/3}L^{2/3}$, with labor fixed at $L$.
2. **Desired capital.** The firm holds the capital stock at which $MPK = uc$, with $uc$ from equation (2), or from equation (3) when there is a tax.
3. **Capital accumulation.** As in the Solow model, capital next year is this year's investment plus the capital that did not wear out:
   $$K_{t+1} = I_t + (1-\bar{d})K_t.$$
4. **Growth.** In the long run the capital stock grows at some rate $g_K \equiv \Delta K_{t+1}/K_t$, set by the growth models of Chapters 5 and 6. We take it as given.

**Our baseline.** We use $\bar{A} = 0.3$ and $L = 1$ alongside $R = 0.02$ and $\bar{d} = 0.08$. These two numbers are our own choice: they make the desired capital stock exactly $K = 1$, with output $Y = 0.3$, so that Figs. 1 and 2 start from round numbers. For the investment rate we use an illustrative $g_K = 0.02$.

#### Solving the model

**Step 1: the marginal product of capital.** The marginal product is the derivative of output with respect to capital, holding labor fixed ([Appendix C.7](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.7-derivatives-and-marginal-thinking)). The power rule brings the exponent $1/3$ down and lowers it by one, to $1/3 - 1 = -2/3$:

$$MPK = \frac{\partial Y}{\partial K} = \frac{1}{3}\bar{A}K^{-2/3}L^{2/3}.$$

This can be written in two useful ways. First, $\bar{A}K^{-2/3}L^{2/3} = \bar{A}K^{1/3}L^{2/3}/K = Y/K$, because dividing by $K$ lowers the exponent of $K$ by one. Second, $K^{-2/3}L^{2/3} = (L/K)^{2/3}$, because two powers with the same exponent can be combined into one power of the ratio ([Appendix C.3](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.3-exponents-and-exponential-growth)). So

$$MPK = \frac{1}{3}\cdot\frac{Y}{K} = \frac{1}{3}\cdot\bar{A}\left(\frac{L}{K}\right)^{2/3}. \tag{4}$$

The exponent $-2/3$ on $K$ is negative, so $MPK$ falls as capital rises: diminishing returns. (The textbook's statement of this production function prints the exponent on labor as $1/2$; with constant returns to scale it must be $2/3$, the value that the formula for $MPK$ uses.)

**Step 2: the desired capital stock.** Set the second form of (4) equal to the user cost and solve for $K$. Multiply both sides by $3/\bar{A}$:

$$\frac{1}{3}\bar{A}\left(\frac{L}{K}\right)^{2/3} = uc \quad\Longrightarrow\quad \left(\frac{L}{K}\right)^{2/3} = \frac{3\,uc}{\bar{A}}.$$

Raise both sides to the power $3/2$, because $\left((L/K)^{2/3}\right)^{3/2} = L/K$, and then turn both sides upside down:

$$\frac{L}{K} = \left(\frac{3\,uc}{\bar{A}}\right)^{3/2} \quad\Longrightarrow\quad K^{\text{desired}} = L\left(\frac{\bar{A}}{3\,uc}\right)^{3/2}. \tag{5}$$

With the baseline, $\bar{A}/(3\,uc) = 0.3/0.3 = 1$, so $K^{\text{desired}} = 1$.

**Step 3: the capital-output ratio.** Use the first form of (4) instead: $\frac{1}{3}\,Y/K = uc$. Multiplying both sides by 3,

$$\frac{Y}{K} = 3\cdot uc. \tag{6}$$

With $uc = 0.10$, output is 30 percent of the capital stock, so the firm holds capital worth $1/0.3 \approx 3.3$ years of output. Equation (6) holds for any $\bar{A}$ and $L$: productivity changes how much capital the firm wants, but not the ratio $Y/K$, which only the user cost sets.

**Step 4: the investment that reaches the desired stock.** Put the firm's current capital $K_t$ and its desired capital $K^{\text{desired}}$ into the accumulation equation, in place of $K_{t+1}$, and solve for investment:

$$I_t = K^{\text{desired}} - (1-\bar{d})K_t.$$

A firm with $K_t = 0.5$ that wants $K = 1$ next year must invest $1 - 0.92\times0.5 = 0.54$: 0.50 to grow, and 0.04 to replace what wears out. When $K^{\text{desired}}$ is below $(1-\bar{d})K_t$, the formula gives negative investment: the firm stops investing and lets depreciation shrink its capital, or sells some of it.

**Step 5: the investment rate.** Subtract $K_t$ from both sides of the accumulation equation to write it as a change, then divide by $K_t$:

$$\Delta K_{t+1} = I_t - \bar{d}K_t \quad\Longrightarrow\quad \frac{\Delta K_{t+1}}{K_t} = \frac{I_t}{K_t} - \bar{d}.$$

Split $I_t/K_t$ into the product $(I_t/Y_t)\cdot(Y_t/K_t)$; the $Y_t$ cancels, so nothing changes, but the investment rate $I_t/Y_t$ now appears:

$$g_K = \frac{I_t}{Y_t}\cdot\frac{Y_t}{K_t} - \bar{d}.$$

Now replace $Y_t/K_t$ by $3\cdot uc$ from equation (6), add $\bar{d}$ to both sides and divide by $3\cdot uc$:

$$\frac{I_t}{Y_t} = \frac{g_K + \bar{d}}{3\cdot uc}. \tag{7}$$

With $g_K = 0.02$, $\bar{d} = 0.08$ and $uc = 0.10$, the investment rate is $0.10/0.30 \approx 33$ percent. With the 35 percent tax, $uc \approx 0.154$ and the rate falls to about 21.7 percent.

*Intuition.* The numerator is the investment needed per unit of capital: enough to grow the stock at $g_K$ and to replace the worn-out part. The denominator turns "per unit of capital" into "per unit of output": a high user cost means little capital per unit of output (equation 6), so the same percentage additions to capital need only a small share of GDP.

#### What the model predicts

Take logs of equations (5) and (7) ([Appendix C.4](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.4-logarithms)):

$$\ln K^{\text{desired}} = \ln L + \tfrac{3}{2}\ln\bar{A} - \tfrac{3}{2}\ln 3 - \tfrac{3}{2}\ln uc, \qquad
\ln\frac{I}{Y} = \ln(g_K + \bar{d}) - \ln 3 - \ln uc.$$

The coefficients are elasticities:

| A 1% rise in ... | changes $K^{\text{desired}}$ by | changes $I/Y$ by | Which curve moves |
|---|---|---|---|
| user cost $uc$ (from $R$, $\bar{d}$, a capital loss or $\tau$) | $-1.5\%$ | $-1\%$ | user-cost line up |
| productivity $\bar{A}$ | $+1.5\%$ | $0$ | $MPK$ curve up |
| labor $L$ | $+1\%$ | $0$ | $MPK$ curve up |
| the after-tax share $1-\tau$ | $+1.5\%$ | $+1\%$ | user-cost line down |

- **Capital reacts more than one for one.** Because $K^{\text{desired}}$ is proportional to $uc^{-3/2}$, the 35 percent tax cuts desired capital by the factor $0.65^{3/2} \approx 0.52$, almost by half. The flat tail of the $MPK$ curve is the reason: when returns diminish slowly, a small rise in the required return removes a lot of capital.
- **Productivity and labor raise capital, not the investment rate.** They shift the $MPK$ curve up, so the firm wants more capital, but equation (7) does not contain them: output rises in proportion and $I/Y$ stays the same.
- **Depreciation has two effects.** A higher $\bar{d}$ raises the replacement investment in the numerator of (7) and also the user cost in the denominator. Writing $uc = R + \bar{d} - \Delta p_k/p_k$ and taking the derivative, the net effect on $I/Y$ has the sign of $R - \Delta p_k/p_k - g_K$. In our baseline $R = g_K = 0.02$ with no capital gain, so the two effects cancel exactly and $I/Y = 1/3$, capital's exponent, whatever $\bar{d}$ is.
- **A theory of the saving rate.** The Solow model of Chapter 5 took the investment rate $\bar{s}$ as given. Equation (7) explains it: taxes, interest rates and depreciation move it through the user cost. Combined with the growth models (which set $g_K$) and the Euler equation of [Chapter 16](https://quarcs-lab.github.io/macro102-site/notebooks/ch16-consumption.html) (which sets the long-run interest rate), equations (6) and (7) close a long-run model of a closed economy: the capital-output ratio, the investment rate and so the consumption share of GDP.
- **Reaching the target takes time.** Real firms face installation and adjustment costs, so they move toward $K^{\text{desired}}$ over several years, not in one jump.

### The stock market: pricing a share

#### Assumptions

1. **Two uses of savings.** A saver can deposit money in a bank at the interest rate $R$, or buy one share of a firm's stock at the price $p_s$, hold it for a year and sell it.
2. **The share's two payoffs.** During the year the firm pays a **dividend**, a share of its profits, to each shareholder. At the end of the year the share sells at a new price, $p_s + \Delta p_s$.
3. **No risk.** The saver knows the dividend and the future price. Real stocks are risky, and much of finance studies how risk affects prices; here we leave it out.
4. **Constant growth.** In the last step the dividend grows at a constant rate $g$ every year, with $g < R$.

#### Solving the model

**Step 1: the arbitrage equation for a stock.** Invest $p_s$ dollars in each option. The bank pays $R\cdot p_s$; the share pays its dividend plus its capital gain. With no risk, both must pay the same, otherwise everyone would buy only the better one and its price would move:

$$\underbrace{R\cdot p_s}_{\text{bank}} = \underbrace{\text{dividend} + \Delta p_s}_{\text{share}}.$$

The dividend plays the role that $MPK$ played for a machine: the payoff the asset brings even if its price does not change.

**Step 2: returns in percent.** Divide both sides by $p_s$:

$$R = \underbrace{\frac{\text{dividend}}{p_s}}_{\text{dividend yield}} + \underbrace{\frac{\Delta p_s}{p_s}}_{\text{capital gain, in percent}}. \tag{8}$$

The interest rate on the left must equal the stock's total return on the right, the sum of the **dividend yield** and the percentage capital gain. If the stock paid more, savers would buy it, its price would rise today, and both terms on the right would fall until the gap closed.

**Step 3: solve for the price.** Subtract the capital gain from both sides of (8), $R - \Delta p_s/p_s = \text{dividend}/p_s$, then turn both sides upside down and multiply by the dividend:

$$p_s = \frac{\text{dividend}}{R - \Delta p_s/p_s}.$$

**Step 4: constant dividend growth.** Suppose the dividend grows at $g$ and guess that the price is always a fixed multiple of the current dividend. Then the price grows at the same rate as the dividend: $\Delta p_s/p_s = g$. Putting this into step 3:

$$p_s = \frac{\text{dividend}}{R - g}. \tag{9}$$

The guess is consistent. Write the arbitrage equation of step 1 as $p_{s,t+1} = (1+R)\,p_{s,t} - \text{dividend}_t$. If $p_{s,t} = \text{dividend}_t/(R-g)$, then

$$p_{s,t+1} = \text{dividend}_t\left[\frac{1+R}{R-g} - \frac{R-g}{R-g}\right] = \text{dividend}_t\,\frac{1+g}{R-g} = \frac{\text{dividend}_{t+1}}{R-g},$$

so the same formula holds next year, and the year after.

*Intuition: a present value.* Equation (9) is the **present discounted value** of all future dividends ([Appendix C.12](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.12-present-value-and-discounting)). A dividend $D$ paid one year from now, growing at $g$ after that, is worth today

$$\frac{D}{1+R} + \frac{D(1+g)}{(1+R)^2} + \frac{D(1+g)^2}{(1+R)^3} + \dots = \frac{D}{1+R}\cdot\frac{1}{1 - \frac{1+g}{1+R}} = \frac{D}{1+R}\cdot\frac{1+R}{R-g} = \frac{D}{R-g},$$

using the sum of a geometric series, $1 + x + x^2 + \dots = 1/(1-x)$ for $x = (1+g)/(1+R) < 1$. A constant dividend is discounted by $R$; a growing one by the smaller number $R - g$. With $g = 0$ and $R = 0.05$, a share that pays 3 dollars a year is worth $3/0.05 = 60$ dollars. If the dividend grows at $g = 0.02$, the share is worth $3/0.03 = 100$ dollars.

**Step 5: the price-earnings ratio.** Divide both sides of (9) by the firm's earnings (its accounting profits per share). The share of earnings paid out as dividends is the **payout ratio**:

$$\frac{p_s}{\text{earnings}} = \frac{\text{dividend}/\text{earnings}}{R - g}. \tag{10}$$

With a payout ratio of one half, $R = 0.05$ and $g = 0.02$ (the starting values of Fig. 3), the model's P/E ratio is $0.5/0.03 \approx 16.7$.

#### What the model predicts

- **Signs.** The price and the P/E ratio rise with the dividend (or the payout ratio) and with expected growth $g$, and fall with the interest rate $R$.
- **Sensitivity.** In logs, $\ln p_s = \ln(\text{dividend}) - \ln(R-g)$, so the price has an elasticity of $-1$ with respect to the gap $R - g$. When the gap is small, a small change in it is a large *percentage* change: cutting $R - g$ from 3 to 2 percentage points, a fall of one third, raises the price by half ($0.03/0.02 = 1.5$). A P/E ratio of 40 with a payout of one half needs a gap of only $0.5/40 = 1.25$ points.
- **Bubbles are hard to spot.** Turning (10) around gives the gap the market's price implies: $R - g = \text{payout}/(P/E)$. A high P/E ratio is either a bubble or a sign that savers expect faster growth or lower interest rates, and the data cannot easily tell the two apart (Fig. 3).
- **Efficient markets.** If prices already include everything that is known about future dividends, only news that nobody expected can move them. Good and bad surprises are equally likely, so tomorrow's price change cannot be predicted from today's information: the price follows a **random walk**. The evidence is that markets come close to this benchmark: most actively managed mutual funds fail to beat a simple index fund for long. But prices swing more than dividends seem to justify, and bubbles appear from time to time.

### Tobin's q: the stock market and investment

#### Definitions

James Tobin compared two values of the same firm: what the stock market says the firm is worth, $V$, the present value of all its future profits, and what it would cost to buy its capital new, $p_kK$. Their ratio is **Tobin's $q$**:

$$q = \frac{V}{p_kK} = \frac{\text{stock market value of the firm}}{\text{replacement cost of its capital}}. \tag{11}$$

#### Solving the model

**Step 1: the value of a firm that owns only capital.** Take a firm with the Cobb-Douglas technology above, with $p_k = 1$, no taxes and a constant price of capital. Capital is paid its marginal product, so the firm's capital income is $MPK\cdot K$; with the exponent $1/3$ this is exactly a third of output. Each year it spends $\bar{d}K$ to replace the capital that wears out, which keeps $K$ constant, and pays the rest to its shareholders. Its dividend is constant, $(MPK - \bar{d})K$, so equation (9) with $g = 0$ prices the firm:

$$V = \frac{(MPK - \bar{d})K}{R}.$$

**Step 2: divide by the value of capital.** With $p_k = 1$, equation (11) gives

$$q = \frac{MPK - \bar{d}}{R}. \tag{12}$$

**Step 3: link to the user cost.** $q = 1$ exactly when $MPK - \bar{d} = R$, that is when $MPK = R + \bar{d}$, the user cost of equation (2) with no capital gain. So $q$ above 1 is the stock market's way of saying that $MPK$ is above the user cost:

$$q > 1 \iff MPK > uc \iff \text{the firm should invest}.$$

In our baseline the firm that already holds its desired capital, $K = 1$, has $MPK = 0.10$ and $q = (0.10 - 0.08)/0.02 = 1$. A firm with only $K = 0.5$ has $MPK \approx 0.159$ and $q \approx 3.9$: each dollar of new capital adds almost four dollars of market value, so it pays to expand.

*Intuition.* When $q > 1$ the firm could sell new shares, buy capital with the money, and be worth more than the capital cost: investing creates value. When $q < 1$ the firm is worth less than its machines, and its owners would gain by selling some of them.

#### What the model predicts

- **$q$ near 1.** With no costs of adjusting the capital stock, firms invest until $MPK = uc$ at every moment, so $q = 1$. When installing capital is slow and costly (stopping a production line, building a new plant), firms close the gap gradually, and $q$ can stay above or below 1 for a while.
- **$q$ predicts investment.** Firms with high $q$ should invest, and those with low $q$ should shrink.
- **In the data, $q$ is often well above 1**, because firms also own assets that are not in $K$: brand names, patents and know-how, which the stock market values.
- **Financial frictions.** Studies of many firms find that tax changes move investment through $q$ and the user cost, as the theory says. But a firm's own cash flow and its access to bank loans and financial markets also matter, which the simple model leaves out.

### Housing: the arbitrage equation for a house

#### Assumptions

1. **The investor.** A saver can keep her down payment in the bank at the interest rate $R$, or use it to buy a house (a condominium, say) at the price $P^{\text{house}}$, rent it out for a year, and sell it.
2. **Leverage.** The down payment is a fraction $\bar{x}$ of the price, and she borrows the rest, $(1-\bar{x})P^{\text{house}}$, with a mortgage at the rate $R$. The leverage ratio, the price of the house per dollar of her own money, is $1/\bar{x}$.
3. **Taxes.** Mortgage interest is deductible at the tax rate $\tau$, so a dollar of interest costs her only $1-\tau$ dollars.
4. **The house.** It earns a rent, wears out at the rate $\bar{d}$ of its value, and its price changes by $\Delta P^{\text{house}}$ during the year. There is no risk and there are no transaction costs.

#### Solving the model

**Step 1: the arbitrage equation.** The bank pays $R$ on the down payment. The house pays the rent, minus depreciation, plus the capital gain, minus the after-tax interest on the mortgage:

$$R\cdot\bar{x}P^{\text{house}} = \text{rent} - \bar{d}P^{\text{house}} + \Delta P^{\text{house}} - R(1-\tau)(1-\bar{x})P^{\text{house}}.$$

**Step 2: divide by the price.** Every term except the rent contains $P^{\text{house}}$:

$$R\bar{x} = \frac{\text{rent}}{P^{\text{house}}} - \bar{d} + \frac{\Delta P^{\text{house}}}{P^{\text{house}}} - (1-\tau)R(1-\bar{x}).$$

**Step 3: collect the interest terms.** Move everything except the rent ratio to the left. Expanding $(1-\tau)R(1-\bar{x}) = (1-\tau)R - (1-\tau)R\bar{x}$, the two terms in $R\bar{x}$ combine to $R\bar{x} - (1-\tau)R\bar{x} = \tau\bar{x}R$:

$$\frac{\text{rent}}{P^{\text{house}}} = \tau\bar{x}R + (1-\tau)R + \bar{d} - \frac{\Delta P^{\text{house}}}{P^{\text{house}}}.$$

**Step 4: solve for the price.** Turn both sides upside down and multiply by the rent:

$$P^{\text{house}} = \frac{\text{rent}}{\tau\bar{x}R + (1-\tau)R + \bar{d} - \Delta P^{\text{house}}/P^{\text{house}}}. \tag{13}$$

*Intuition.* The denominator is the **user cost of a house**, the analog of equation (2): the interest on the down payment and on the loan (lowered by the tax deduction), plus depreciation, minus the expected price gain. The price is the present value of the rents, discounted at that user cost. With no tax, $\tau = 0$, the denominator is simply $R + \bar{d} - \Delta P^{\text{house}}/P^{\text{house}}$, the user cost of any capital good.

#### What the model predicts

The price rises when the denominator of (13) falls. Taking the derivative of the denominator with respect to each parameter:

| A rise in ... | changes the denominator by | House price |
|---|---|---|
| rent | no change (the numerator rises) | rises in proportion |
| interest rate $R$ | $\tau\bar{x} + 1 - \tau$ per point | falls |
| depreciation $\bar{d}$ | $+1$ per point | falls |
| expected price growth $\Delta P/P$ | $-1$ per point | rises |
| down payment share $\bar{x}$ | $+\tau R$ per unit | falls |
| tax rate $\tau$ | $-(1-\bar{x})R$ per unit | rises |

- **Leverage matters, but only a little.** A smaller down payment (more leverage) raises the price, but its effect is multiplied by $\tau R$, a small number: with $\tau = 0.3$ and $R = 0.04$ it is 0.012, so cutting $\bar{x}$ by 10 points lowers the denominator by only 0.0012.
- **Expectations matter a lot.** One more point of expected price growth lowers the denominator by a full 0.01, about eight times as much as that 10-point cut in the down payment. If rising prices make people expect further rises, the price can climb on its own: a housing bubble that feeds on itself. As with stocks, it is hard to tell such a bubble from a genuine change in rents, interest rates or credit.
- **Inventories, the last part of investment,** follow the same logic of comparing returns, but the chapter treats them in words: firms keep goods in the pipeline of production and on the shelf to avoid running out, and both motives make inventory investment rise in booms and fall in recessions (Fig. 4).

### The model in code

We now turn the key results into Python functions, defined once and reused by every figure. The cells after them check the algebra above with numbers.

> **Python note.** `def` creates a *function*: a named recipe that takes inputs (the
> *arguments*) and gives back a result with `return`. Writing `gpk=0.0` gives the argument a
> *default value*, used whenever we do not pass one. The text in triple quotes is a short
> description of what the function does. `**` means "to the power of". Names in capital letters,
> such as `A_BAR`, are constants that we set once and reuse.

In [3]:
A_BAR, L_BAR = 0.3, 1.0          # illustrative productivity and labor: the baseline desired capital is K = 1


def user_cost(R, d, gpk=0.0, tau=0.0):
    """User cost of capital: (R + d - gpk) / (1 - tau)   (equations 2 and 3; gpk is the capital gain)."""
    return (R + d - gpk) / (1 - tau)


def mpk(K, A=A_BAR, L=L_BAR):
    """Marginal product of capital: MPK = (1/3) A (L/K)^(2/3)   (equation 4)."""
    return A / 3 * (L / K) ** (2 / 3)


def desired_capital(uc, A=A_BAR, L=L_BAR):
    """Desired capital stock, where MPK = uc: K = L (A / (3 uc))^(3/2)   (equation 5)."""
    return L * (A / (3 * uc)) ** (3 / 2)


def investment_rate(uc, g_K=0.02, d=0.08):
    """Investment rate: I/Y = (g_K + d) / (3 uc)   (equation 7)."""
    return (g_K + d) / (3 * uc)

In [4]:
def stock_price(dividend, R, g=0.0):
    """Price of a stock: dividend / (R - g)   (equation 9; dividends grow at rate g)."""
    return dividend / (R - g)


def pe_ratio(payout, R, g):
    """Price-earnings ratio: (dividend / earnings) / (R - g)   (equation 10)."""
    return payout / (R - g)


def tobin_q(K, R=0.02, d=0.08, A=A_BAR):
    """Tobin's q of a firm that owns only capital: (MPK - d) / R   (equation 12)."""
    return (mpk(K, A) - d) / R

**Check 1: the user cost and the arbitrage equation.** At the desired capital stock, a dollar in capital must earn exactly what a dollar in the bank earns, with and without the tax.

> **Python note.** An *f-string* such as `f"{uc:.1%}"` inserts the value of a variable into a
> piece of text; `:.1%` shows a share as a percentage with one decimal (0.1 becomes 10.0%), and
> `:.4f` shows four decimals.

In [5]:
uc_base = user_cost(R=0.02, d=0.08)
uc_tax = user_cost(R=0.02, d=0.08, tau=0.35)
K_base, K_tax = desired_capital(uc_base), desired_capital(uc_tax)
print(f"User cost, no tax (equation 2):       {uc_base:.1%}")
print(f"User cost, 35% tax (equation 3):      {uc_tax:.1%}")
print(f"No tax:  bank pays R = 0.0200, capital pays MPK - d = {mpk(K_base) - 0.08:.4f}")
print(f"Tax:     bank pays R = 0.0200, capital pays (1 - tau) MPK - d = {(1 - 0.35) * mpk(K_tax) - 0.08:.4f}")
print(f"A capital gain of 2%: user cost {user_cost(0.02, 0.08, gpk=0.02):.1%}; a loss of 4%: {user_cost(0.02, 0.08, gpk=-0.04):.1%}")

User cost, no tax (equation 2):       10.0%
User cost, 35% tax (equation 3):      15.4%
No tax:  bank pays R = 0.0200, capital pays MPK - d = 0.0200
Tax:     bank pays R = 0.0200, capital pays (1 - tau) MPK - d = 0.0200
A capital gain of 2%: user cost 8.0%; a loss of 4%: 14.0%


**Check 2: desired capital, the capital-output ratio and the elasticity.** Equation (5) must give a capital stock at which $MPK$ equals the user cost, and there $Y/K$ must equal $3\cdot uc$ (equation 6). Raising the user cost by 1 percent should lower desired capital by about 1.5 percent.

In [6]:
Y_base = A_BAR * K_base ** (1 / 3) * L_BAR ** (2 / 3)
print(f"Desired capital K = {K_base:.3f}, MPK there = {mpk(K_base):.4f}, user cost = {uc_base:.4f}")
print(f"Y/K = {Y_base / K_base:.4f}   and   3 x uc = {3 * uc_base:.4f}")
change_K = desired_capital(uc_base * 1.01) / K_base - 1
print(f"1% higher user cost: desired capital changes by {100 * change_K:+.3f}%   (elasticity -1.5)")
print(f"35% tax: K falls by the factor {K_tax / K_base:.4f}   and   0.65^(3/2) = {0.65 ** 1.5:.4f}")

Desired capital K = 1.000, MPK there = 0.1000, user cost = 0.1000
Y/K = 0.3000   and   3 x uc = 0.3000
1% higher user cost: desired capital changes by -1.481%   (elasticity -1.5)
35% tax: K falls by the factor 0.5240   and   0.65^(3/2) = 0.5240


**Check 3: the investment rate, two ways.** Let a firm hold its desired capital and grow it at $g_K = 2$ percent. We compute the investment it needs from the accumulation equation, divide by output, and compare with equation (7).

In [7]:
for tau in [0.0, 0.35]:
    uc = user_cost(0.02, 0.08, tau=tau)
    K_now = desired_capital(uc)
    K_next = 1.02 * K_now                          # capital grows at g_K = 2 percent
    I = K_next - (1 - 0.08) * K_now                # accumulation equation solved for investment
    Y = A_BAR * K_now ** (1 / 3)
    print(f"tau = {tau:.2f}: I/Y = {I / Y:.4f} (accumulation), {investment_rate(uc):.4f} (equation 7)")

tau = 0.00: I/Y = 0.3333 (accumulation), 0.3333 (equation 7)
tau = 0.35: I/Y = 0.2167 (accumulation), 0.2167 (equation 7)


Equation (7) also says that, with $R = g_K$ and no capital gain, depreciation does not change the investment rate.

In [8]:
for d in [0.05, 0.08, 0.12]:
    print(f"d = {d:.2f}: I/Y = {investment_rate(user_cost(0.02, d), g_K=0.02, d=d):.4f}")

d = 0.05: I/Y = 0.3333
d = 0.08: I/Y = 0.3333
d = 0.12: I/Y = 0.3333


**Check 4: the stock price is a present value.** We add up the discounted dividends of the next 2,000 years, one year at a time, and compare the sum with equation (9). Then we confirm that the arbitrage equation makes the price grow at the rate $g$.

> **Python note.** `range(1, 2001)` counts 1, 2, ..., 2000, so the `for` loop runs its indented lines once
> for each year. `pv = pv + ...` adds one more term to the running total.

In [9]:
D, R, g = 3.0, 0.05, 0.02                          # first dividend (dollars), interest rate, growth
pv = 0.0
for t in range(1, 2001):
    pv = pv + D * (1 + g) ** (t - 1) / (1 + R) ** t    # dividend of year t, discounted to today
print(f"Sum of discounted dividends: {pv:.4f}   equation (9): {stock_price(D, R, g):.4f}")
print(f"No growth: {stock_price(D, R):.2f} dollars")

p_now = stock_price(D, R, g)
p_next = (1 + R) * p_now - D                       # arbitrage: R p = dividend + change in p
print(f"Price growth implied by arbitrage: {p_next / p_now - 1:.4f}   dividend growth g = {g:.4f}")
print(f"Model P/E ratio with a payout of 0.5: {pe_ratio(0.5, R, g):.1f}")

Sum of discounted dividends: 100.0000   equation (9): 100.0000
No growth: 60.00 dollars
Price growth implied by arbitrage: 0.0200   dividend growth g = 0.0200
Model P/E ratio with a payout of 0.5: 16.7


**Check 5: Tobin's $q$.** A firm at its desired capital should have $q = 1$, and its market value from equation (9) should equal its capital. A firm below its desired capital should have $q > 1$, and one above it $q < 1$.

In [10]:
for K_firm in [0.5, 1.0, 1.2]:
    V = (mpk(K_firm) - 0.08) * K_firm / 0.02       # dividend (MPK - d) K priced with equation (9), g = 0
    print(f"K = {K_firm:.1f}: MPK = {mpk(K_firm):.4f}, V = {V:.3f}, q = V/K = {V / K_firm:.3f}, "
          f"equation (12): {tobin_q(K_firm):.3f}")

K = 0.5: MPK = 0.1587, V = 1.969, q = V/K = 3.937, equation (12): 3.937
K = 1.0: MPK = 0.1000, V = 1.000, q = V/K = 1.000, equation (12): 1.000
K = 1.2: MPK = 0.0886, V = 0.513, q = V/K = 0.428, equation (12): 0.428


**Check 6: the house price.** We compute the price from equation (13), with the numbers used later in the notebook (a rent of 6,000 dollars a year, $R = 0.04$, $\bar{d} = 0.02$, $\tau = 0.3$, $\bar{x} = 0.2$, expected price growth of 2 percent). Then we put that price back into both sides of the arbitrage equation of step 1: they must match.

In [11]:
rent, R, d, tau, x, gp = 6000, 0.04, 0.02, 0.3, 0.2, 0.02
P = rent / (tau * x * R + (1 - tau) * R + d - gp)  # equation (13)
bank = R * x * P                                   # return from keeping the down payment in the bank
house = rent - d * P + gp * P - R * (1 - tau) * (1 - x) * P    # return from buying the house
print(f"House price: {P:,.0f} dollars")
print(f"Bank: {bank:,.2f} dollars   House: {house:,.2f} dollars")

House price: 197,368 dollars
Bank: 1,578.95 dollars   House: 1,578.95 dollars


### Drawing tools

Figs. 1 and 2 are the same kind of diagram, so we write its drawing steps once:

- `capital_figure` starts a diagram with fixed axes, so that you can see the curves move;
- `add_mpk` draws the marginal product curve, and `add_level` draws a flat user-cost line; both
  write a label on the plot next to the line (its equation or its value), and `add_label` skips a
  label that would sit on top of another one;
- `add_point` marks a desired capital stock with a dashed guide to the axis, and `add_arrow` draws an arrow.

> **Python note.** `np.linspace(0.05, K_max, 300)` creates 300 evenly spaced numbers from 0.05 to `K_max`,
> the right end of the x-axis, which `fig.layout.xaxis.range[1]` reads from the figure.
> Because `mpk` works on a whole array at once, one call gives the whole curve. `go.Figure()`
> starts an empty chart, and each `add_trace` adds one line or set of points. `MPK <= 0.4` is
> true where the curve is inside the axes, `K[inside]` keeps those values, and `[0]` takes the
> first of them. A `for` loop over `fig.layout.annotations` looks at every label already written,
> and `return` inside the loop stops the function early.

In [12]:
K_RANGE, RETURN_RANGE = [0, 3], [0, 0.4]          # fixed axes of Figs. 1 and 2


def capital_figure(title):
    """An empty diagram: capital on the x-axis, returns per unit of capital (percent) on the y-axis."""
    fig = go.Figure()
    fig.update_layout(title=title, showlegend=False,
                      xaxis=dict(title="Capital, K", range=K_RANGE, tickvals=[0.5, 1, 1.5, 2, 2.5, 3], zeroline=False),
                      yaxis=dict(title="Return per unit of capital", range=RETURN_RANGE,
                                 tickvals=[0, 0.1, 0.2, 0.3, 0.4], tickformat=".0%", zeroline=False))
    return fig


def add_label(fig, x, y, text, color, xanchor="left"):
    """Write a line name at (x, y), unless another name already sits within 2 points of it."""
    for old in fig.layout.annotations:
        if old.showarrow is False and abs(old.x - x) < 0.5 and abs(old.y - y) < 0.02:
            return
    fig.add_annotation(x=x, y=y, text=text, showarrow=False, xanchor=xanchor, yanchor="bottom",
                       font=dict(color=color, size=14))

In [13]:
def add_mpk(fig, A=A_BAR):
    """Draw the MPK curve of equation (4) and write its name next to its top-left end."""
    K_max = fig.layout.xaxis.range[1]                     # the right end of the x-axis
    K = np.linspace(0.05, K_max, 300)
    MPK = mpk(K, A)
    inside = MPK <= RETURN_RANGE[1]                       # the part of the curve inside the axes
    fig.add_trace(go.Scatter(x=K[inside], y=MPK[inside], mode="lines", name="MPK", line=dict(color=BLUE, width=3)))
    add_label(fig, K[inside][0] + 0.05, MPK[inside][0] - 0.03, "MPK = (A/3)·(L/K)<sup>2/3</sup>", BLUE)


def add_level(fig, uc, label, color, dash="solid"):
    """Draw a flat user-cost line at the height uc and write its name at its right end."""
    K_max = fig.layout.xaxis.range[1]
    fig.add_trace(go.Scatter(x=[0, K_max], y=[uc, uc], mode="lines", name=label, line=dict(color=color, width=3, dash=dash)))
    add_label(fig, K_max, uc, label, color, xanchor="right")


def add_point(fig, K, uc, label, anchor="left"):
    """Mark (K, uc) with a dot and a dashed guide down to the x-axis, labelled next to the guide."""
    fig.add_trace(go.Scatter(x=[K, K], y=[0, uc], mode="lines", hoverinfo="skip",
                             line=dict(color="black", width=1, dash="dash")))
    fig.add_trace(go.Scatter(x=[K], y=[uc], mode="markers", name=label, marker=dict(color="black", size=11)))
    fig.add_annotation(x=K, y=0, text=label, showarrow=False, xanchor=anchor, yanchor="bottom", font=dict(size=14))


def add_arrow(fig, x0, y0, x1, y1, text=""):
    """Draw an arrow from (x0, y0) to (x1, y1), with an optional text at its tail; nothing if the points almost coincide."""
    if abs(x1 - x0) > 0.03 or abs(y1 - y0) > 0.005:
        fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref="x", yref="y", axref="x", ayref="y", text=text,
                           showarrow=True, arrowhead=3, arrowwidth=2, arrowcolor="black")

## How firms choose their capital stock

The firm's decision fits in one diagram: the marginal product of capital falls as capital rises, and the firm stops where it meets the user cost. A corporate tax raises that cost and shrinks the desired capital stock.

### Fig. 1: How much should a firm invest?

**Goal.** Find the desired capital stock of a firm, and see how the interest rate, depreciation,
capital gains and productivity change it.

The marginal product of capital falls as the firm adds capital: the first pizza oven adds a lot of
pizzas, the tenth adds few. The user cost does not depend on how much capital the firm has, so it is
a flat line. The firm keeps investing as long as one more unit of capital earns more than it costs,
so it stops where the two lines cross, at $K^{\text{desired}}$.

Investment is what moves the firm toward that point. The capital accumulation equation says that
next year's capital is this year's investment plus the capital that did not wear out:

$$K_{t+1} = I_t + (1-\bar{d})\,K_t$$

So a young firm that starts with $K_0$ and wants $K^{\text{desired}}$ next year must invest
$I = K^{\text{desired}} - (1-\bar{d})K_0$. The title shows this number. The arrow points from $K_0$ to
the desired capital stock.

> **Python note.** `if K_star > 2.6:` runs the indented lines below it only when the condition is true. Here
> it widens the x-axis (`update_xaxes`) in the rare case of a very low user cost, when the desired capital
> stock would otherwise fall off the chart; `tickvals=None` lets Plotly choose the tick marks. `round(x, 2)`
> rounds a number to two decimals, and adding `0.0` turns a rounded `-0.0` into `0.0`, so the title never shows "-0.00".

In [14]:
def desired_capital_diagram(R=0.02, d=0.08, gpk=0.0, A=A_BAR, K0=0.5):
    """Fig. 17.1: invest until the marginal product of capital falls to the user cost."""
    uc = user_cost(R, d, gpk)
    K_star = desired_capital(uc, A)
    I = round(K_star - (1 - d) * K0, 2) + 0.0        # investment that reaches K_star next year (capital accumulation)
    fig = capital_figure(f"User cost {uc:.1%}  |  desired capital {K_star:.2f}  |  investment from K0 = {K0:.1f}: {I:+.2f}")
    if K_star > 2.6:                                 # a very low user cost: widen the x-axis so K desired stays on it
        fig.update_xaxes(range=[0, 1.15 * K_star], tickvals=None)
    add_mpk(fig, A)
    add_level(fig, uc, f"R + d − Δp/p = {uc:.1%}", TEAL)
    add_point(fig, K_star, uc, "K<sup>desired</sup>")
    add_arrow(fig, K0, 0.03, K_star, 0.03, text="K<sub>0</sub>")
    return fig

Now the interactive version. Move the sliders and watch where the two lines cross.

> **Python note.** `interact` turns each argument of a function into a slider and calls the
> function again every time you move one. The `;` at the end only hides an extra line of text.

In [15]:
interact(desired_capital_diagram,
         R=FloatSlider(value=0.02, min=0.01, max=0.08, step=0.01, description="interest rate R", style=WIDE),
         d=FloatSlider(value=0.08, min=0.05, max=0.15, step=0.01, description="depreciation rate d", style=WIDE),
         gpk=FloatSlider(value=0.0, min=-0.04, max=0.04, step=0.01, description="capital gain Δp/p", style=WIDE),
         A=FloatSlider(value=0.3, min=0.15, max=0.45, step=0.05, description="productivity A", style=WIDE),
         K0=FloatSlider(value=0.5, min=0.1, max=2.5, step=0.1, description="starting capital K0", style=WIDE));

In [16]:
# @title Website version: Fig. 1 with a precomputed slider for the interest rate
slider_figure(desired_capital_diagram, "R", [0.01, 0.02, 0.03, 0.04, 0.05, 0.06], start=0.02,
              label="interest rate R")

**Source.** Adapted from Fig. 17.1 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- In the baseline the user cost is 10 percent and the desired capital stock is 1. A firm that starts
  with $K_0 = 0.5$ must invest 0.54: half a unit to grow, and 0.04 to replace the capital that wears out.
- A higher interest rate or faster depreciation raises the user cost and moves $K^{\text{desired}}$ to
  the left. Because the MPK curve flattens out, a rise of 1 point in the user cost lowers desired
  capital by more when the user cost is low.
- A **capital gain** lowers the user cost. Structures, whose land gets scarcer, often gain in value;
  computers lose value quickly, so their user cost is high. Set $\Delta p/p = -0.04$: desired capital
  falls to 0.60.
- Higher productivity $\bar{A}$ shifts the MPK curve up, so the firm wants more capital at the same user cost.

**Your turn.** A firm can buy either computers, whose price falls by 4 percent a year, or a building,
whose price rises by 2 percent a year. With $R = 0.02$ and $\bar{d} = 0.08$ for both, how much more
capital does the firm want in buildings than in computers? Use `user_cost` and `desired_capital` in the
cell below.

<details><summary>Hint</summary>

The capital gain enters the user cost with a minus sign: `user_cost(0.02, 0.08, gpk=-0.04)` for the
computers and `gpk=0.02` for the building. Then divide the two desired capital stocks.

</details>

In [17]:
# Your turn: try it here

### Fig. 2: An increase in the corporate income tax

**Goal.** See why a corporate income tax lowers the capital stock that firms want.

The tax takes a share $\tau$ of what each unit of capital earns, so the firm keeps only
$(1-\tau)MPK$. To make an investment worthwhile, the marginal product before tax must now cover the
user cost divided by $1-\tau$ (equation 3). The user-cost line shifts up, and the firm moves up
its MPK curve to a smaller capital stock. With the textbook's numbers, a 35 percent tax raises the
user cost from 10 to about 15 percent.

In [18]:
def corporate_tax_diagram(tau=0.35, R=0.02, d=0.08):
    """Fig. 17.2: a corporate income tax raises the user cost and lowers the desired capital stock."""
    uc0, uc1 = user_cost(R, d), user_cost(R, d, tau=tau)
    K0, K1 = desired_capital(uc0), desired_capital(uc1)
    fig = capital_figure(f"User cost {uc0:.1%} → {uc1:.1%}  |  desired capital {K0:.2f} → {K1:.2f} ({K1 / K0 - 1:+.0%})")
    add_mpk(fig)
    add_level(fig, uc1, f"τ = {tau:.0%}: {uc1:.1%}", TEAL)
    add_level(fig, uc0, f"τ = 0: {uc0:.1%}", GRAY, dash="dash")      # the old line, dashed
    add_point(fig, K0, uc0, "K<sup>initial</sup>")
    add_point(fig, K1, uc1, "K<sup>desired</sup>", anchor="right")
    add_arrow(fig, 0.06, uc0, 0.06, uc1)                              # the user cost shifts up
    add_arrow(fig, K0, 0.06, K1, 0.06)                                # desired capital falls
    return fig

The economy starts with no tax. Choose a tax rate, then press **Introduce the tax!**

> **Python note.** `interact_manual` works like `interact`, but adds a button: nothing is redrawn until
> you press it. `.options(manual_name=...)` sets the text on the button.

In [19]:
introduce = interact_manual.options(manual_name="Introduce the tax!")
introduce(corporate_tax_diagram,
          tau=FloatSlider(value=0.35, min=0.0, max=0.5, step=0.05, description="corporate tax rate τ", style=WIDE),
          R=FloatSlider(value=0.02, min=0.01, max=0.06, step=0.01, description="interest rate R", style=WIDE),
          d=FloatSlider(value=0.08, min=0.05, max=0.12, step=0.01, description="depreciation rate d", style=WIDE));

In [20]:
# @title Website version: Fig. 2 with a precomputed slider for the corporate tax rate
slider_figure(corporate_tax_diagram, "tau", [0.0, 0.1, 0.2, 0.21, 0.3, 0.35, 0.4, 0.5], start=0.35,
              label="corporate tax rate τ")

**Source.** Adapted from Fig. 17.2 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- With $\tau = 0.35$ the user cost rises from 10.0 to 15.4 percent, and the desired capital stock falls
  by about half, from 1.00 to 0.52.
- The capital stock falls by much more than 35 percent. Desired capital is proportional to
  $uc^{-3/2}$, so it falls by the factor $(1-\tau)^{3/2} = 0.65^{1.5} \approx 0.52$.
- A higher interest rate moves *both* user-cost lines up, and the tax multiplies every change in
  $R$ by $1/(1-\tau)$.

#### Key numbers: from desired capital to investment

After the tax, the firm has more capital than it wants. The capital accumulation equation says it can simply stop
investing and let depreciation do the work. How many years does that take?

> **Python note.** A `while` loop repeats its indented lines as long as its condition is true. Here
> each pass is one year in which the capital stock shrinks by the depreciation rate, and `years += 1`
> adds one to the counter.

In [21]:
K_initial = desired_capital(user_cost(0.02, 0.08))
K_target = desired_capital(user_cost(0.02, 0.08, tau=0.35))
K, years = K_initial, 0
while K > K_target:
    K = (1 - 0.08) * K                  # no new investment: capital accumulation with I = 0
    years += 1
print(f"Capital falls from {K_initial:.2f} to {K_target:.2f}: {years} years with no investment at all")

Capital falls from 1.00 to 0.52: 8 years with no investment at all


In the long run the tax also lowers the **investment rate**. Equation (7) says that, for a given
growth rate of capital $g_K$, the investment rate is inversely proportional to the user cost. We use
$g_K = 2$ percent, an illustrative long-run growth rate:

In [22]:
s_before = investment_rate(user_cost(0.02, 0.08))
s_after = investment_rate(user_cost(0.02, 0.08, tau=0.35))
print(f"Investment rate I/Y without the tax: {s_before:.1%}")
print(f"Investment rate I/Y with a 35% tax:  {s_after:.1%}  (a fall by the factor 1 - tau = {s_after / s_before:.2f})")

Investment rate I/Y without the tax: 33.3%
Investment rate I/Y with a 35% tax:  21.7%  (a fall by the factor 1 - tau = 0.65)


#### Key numbers: Table 17.1, corporate taxes across countries

Table 17.1 of the textbook lists corporate tax rates in 33 countries and the user cost they imply. The
textbook assumes a user cost of 10 percent in the United States, where the tax rate is 21 percent. The
user cost before taxes is then $10 \times (1-0.21) = 7.9$ percent everywhere, and each country's user
cost is $7.9/(1-\tau)$, as in equation (3). We load the OECD tax rates for every year since 2000,
and the table as printed in the textbook.

> **Python note.** `pd.read_csv` reads a data file into a *DataFrame*, a table with named columns, and
> `.head()` shows its first rows. `taxes[taxes.year == year]` keeps only the rows of one year, and
> `.copy()` makes it a new table to which we can add a column. `book.merge(table, on="country")` joins
> two tables by matching the country names, like a lookup in a spreadsheet.

In [23]:
taxes = pd.read_csv(DATA_URL + "corporate_tax.csv")
book = pd.read_csv(DATA_URL + "table17-1-book.csv")
taxes.head()

,country,iso3,year,cit_rate
0,Brazil,BRA,2000,37.0
1,Brazil,BRA,2001,34.0
2,Brazil,BRA,2002,34.0
3,Brazil,BRA,2003,34.0
4,Brazil,BRA,2004,34.0


Each row of `taxes` is one country in one year; `cit_rate` is the statutory corporate income tax rate
of the central government, in percent.

In [24]:
def tax_table(year=2022):
    """Table 17.1 for one year: tax rates and user costs, with the U.S. user cost fixed at 10 percent."""
    table = taxes[taxes.year == year].copy()
    tau_us = table[table.country == "United States"].cit_rate.iloc[0] / 100
    uc_no_tax = 0.10 * (1 - tau_us)                       # the user cost before taxes
    table["user_cost"] = 100 * uc_no_tax / (1 - table.cit_rate / 100)       # equation (3)
    table = book.merge(table[["country", "cit_rate", "user_cost"]], on="country")
    names = {"book_rate": "tax rate, book", "cit_rate": f"tax rate, {year}",
             "book_user_cost": "user cost, book", "user_cost": f"user cost, {year}"}
    return table.rename(columns=names).round(1)


table_2022 = tax_table(2022)
table_2022

,country,"tax rate, book","user cost, book","tax rate, 2022","user cost, 2022"
0,Brazil,34.0,12.0,34.0,12.0
1,Argentina,30.0,11.3,35.0,12.2
2,Democratic Republic of the Congo,30.0,11.3,30.0,11.3
3,Kenya,30.0,11.3,30.0,11.3
4,Nigeria,30.0,11.3,30.0,11.3
5,Australia,30.0,11.3,30.0,11.3
6,Costa Rica,30.0,11.3,30.0,11.3
7,Mexico,30.0,11.3,30.0,11.3
8,Portugal,30.0,11.3,30.0,11.3
9,South Africa,27.0,10.8,28.0,11.0


**Look at the differences.** Three rates of 2022 differ from the textbook. Always ask why before you
use the numbers:

In [25]:
different = table_2022[(table_2022["tax rate, 2022"] - table_2022["tax rate, book"]).abs() > 0.05]
different

,country,"tax rate, book","user cost, book","tax rate, 2022","user cost, 2022"
1,Argentina,30.0,11.3,35.0,12.2
9,South Africa,27.0,10.8,28.0,11.0
30,Chile,10.0,8.8,27.0,10.8


- **Chile:** the textbook's 10 percent is Chile's lower rate for *small businesses*. The statutory rate
  for other companies is 27 percent, the number we keep.
- **South Africa:** the textbook already uses the 27 percent rate that applies from 2023; in 2022 it was 28.
- **Argentina:** since 2021 Argentina taxes profits in brackets of 25, 30 and 35 percent, and the OECD
  reports the top one.

Pick a year in the menu to rebuild the table for that year.

> **Python note.** A `Dropdown` is a menu. `interact` shows the table that `tax_table` returns, and
> builds it again whenever you choose another year.

In [26]:
interact(tax_table, year=Dropdown(options=list(range(2000, 2027)), value=2022, description="year", style=WIDE));

**Source.** Data: OECD, Corporate Tax Statistics, statutory corporate income tax rate of the central government (dataflow `DSD_TAX_CIT@DF_CIT`, measure `CIT`), via the [OECD Data Explorer](https://data-explorer.oecd.org/). The columns marked "book" are the values printed in the textbook, kept for comparison.
Prepared files: [`data/ch17/corporate_tax.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/corporate_tax.csv), [`data/ch17/table17-1-book.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/table17-1-book.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/README.md)).
Table: adapted from Table 17.1 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

The tax rates range from 0 to 34 percent in the textbook, but the user cost only from 7.9 to 12.0
percent. Taxes matter, but a 34-point difference in tax rates raises the user cost only by about half. The
U.S. rate itself was 35 percent until 2017 (the textbook says 34, the rate on middle
incomes) and 21 percent from 2018:

In [27]:
us = taxes[(taxes.country == "United States") & (taxes.year >= 2015)]
us.set_index("year")[["cit_rate"]].T         # .T turns the table on its side, so the years run across

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
cit_rate,35.0,35.0,35.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0


**Your turn.** The Tax Cuts and Jobs Act of 2017 cut the U.S. corporate tax rate from 35 to 21 percent.
By what factor did it raise the desired capital stock, according to Fig. 2's model? First move the
slider to $\tau = 0.35$ and then to $\tau = 0.21$, then compute the ratio with `desired_capital` and
`user_cost`. Finally, run `tax_table(2025)`: which countries changed their tax rate after 2022?

<details><summary>Hint</summary>

Compute `desired_capital(user_cost(0.02, 0.08, tau=0.21)) / desired_capital(user_cost(0.02, 0.08, tau=0.35))`.
Because $K^{\text{desired}}$ is proportional to $(1-\tau)^{3/2}$, the answer is $(0.79/0.65)^{3/2}$. For the
second part, compare the columns `tax rate, book` and `tax rate, 2025`, as we did above for 2022.

</details>

In [28]:
# Your turn: try it here

## The stock market

The same comparison of returns tells us what a share of stock is worth. Here we compare the model's price-earnings ratio with more than a century of U.S. stock-market data.

### Fig. 3: The price-earnings ratio in the stock market

**Goal.** Read the price-earnings ratio of the U.S. stock market with the arbitrage equation for stocks.

Robert Shiller's **price-earnings (P/E) ratio** divides the S&P 500 stock price index by the average
real earnings of its companies over the previous ten years. Equation (10) says what the ratio
*should* be: the payout ratio divided by $R-g$. The teal line in the figure is this model ratio. With
the textbook's numbers from the stock example, $R = 5$ percent and $g = 2$ percent, and a payout ratio
of one half, the model ratio is $0.5/0.03 \approx 16.7$, close to the long-run average.

This figure stays U.S.-only: Shiller's monthly data are the only free, long series of a ten-year P/E ratio. Ratios
for other stock markets are sold by commercial data providers and cannot be shared here.

In [29]:
cape = pd.read_csv(DATA_URL + "fig17-3.csv", parse_dates=["date"])
cape.tail(3)

,date,cape,cape_average
1747,2026-08-01,40.9874,17.8044
1748,2026-09-01,40.5809,17.8044
1749,2026-10-01,40.7106,17.8044


Each row is a month. `cape` is the P/E ratio and `cape_average` its average over all months since 1881.
`parse_dates=["date"]` turns the text in the `date` column into real dates.

> **Python note.** `px.area` draws a line and fills the area below it. `cape.iloc[-1]` is the last row
> of the table. `if R > g:` runs the indented lines only when the condition holds; `else:` handles the
> other case. Here it protects us from dividing by zero, because equation (9) has no finite price
> when $g \geq R$. `"Model" if model < 48 else "Model ↑"` is a one-line `if`: it gives the first text when the
> condition holds and the second one otherwise. `min(model, 48)` keeps the label inside the chart.

In [30]:
def pe_chart(R=0.05, g=0.02, payout=0.5):
    """Fig. 17.3: the P/E ratio, its average, and the model ratio payout / (R - g) of equation (10)."""
    last, average = cape.iloc[-1], cape.cape.mean()
    fig = px.area(cape, x="date", y="cape", color_discrete_sequence=["#5aa0d8"],
                  labels={"date": "Year", "cape": "P/E ratio"})
    fig.add_trace(go.Scatter(x=[cape.date.iloc[0], last.date], y=[average, average], mode="lines",
                             name="Average", line=dict(color="black", width=1.5, dash="dash")))
    fig.add_annotation(x="1981-01-01", y=average + 7, text=f"Mean 1881–{last.date.year}: {average:.1f}", showarrow=False, font=dict(size=14))
    model_text = "model: no finite price, because g ≥ R"
    if R > g:
        model = pe_ratio(payout, R, g)
        fig.add_trace(go.Scatter(x=[cape.date.iloc[0], last.date], y=[model, model], mode="lines",
                                 name="Model, equation (10)", line=dict(color=TEAL, width=3)))
        model_text = f"model: {payout:.2f} / ({R:.1%} − {g:.2%}) = {model:.1f}"
        label = "Model" if model < 48 else "Model ↑"      # a model line above the chart is not visible: point up
        fig.add_annotation(x=last.date, y=min(model, 48), text=label, showarrow=False, xanchor="left",
                           xshift=6, font=dict(color=TEAL, size=14))
    fig.update_layout(title=f"P/E ratio {last.cape:.1f} in {last.date:%b %Y}  |  {model_text}",
                      xaxis=dict(range=["1881-01-01", "2041-01-01"]),     # room for the label on the right
                      yaxis=dict(range=[0, 50], tickvals=[0, 10, 20, 30, 40, 50]), showlegend=False)
    return fig

In [31]:
interact(pe_chart,
         R=FloatSlider(value=0.05, min=0.03, max=0.08, step=0.005, description="interest rate R", style=WIDE, readout_format=".3f"),
         g=FloatSlider(value=0.02, min=0.0, max=0.04, step=0.0025, description="dividend growth g", style=WIDE, readout_format=".4f"),
         payout=FloatSlider(value=0.5, min=0.3, max=0.8, step=0.05, description="payout ratio", style=WIDE));

In [32]:
# @title Website version: Fig. 3 with a precomputed slider for dividend growth
slider_figure(pe_chart, "g", [0.0, 0.01, 0.02, 0.025, 0.03, 0.035, 0.0375], start=0.02,
              label="dividend growth g")

**Source.** Data: [Robert J. Shiller, Online Data, cyclically adjusted price-earnings ratio (CAPE) of the S&P composite index (file `ie_data.xls`)](https://shillerdata.com/), formerly at [econ.yale.edu](http://www.econ.yale.edu/~shiller/data.htm). The model line is our own, from equation (10).
Prepared file: [`data/ch17/fig17-3.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/fig17-3.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/README.md)).
Chart: inspired by Fig. 17.3 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- The ratio is very volatile. It peaked in September 1929, before the crash, and reached its record,
  44.2, in December 1999 at the top of the dot-com boom (the textbook dates this peak "in 2001"; by
  2001 the ratio had already fallen to between 28 and 37). It was below 7 in 1920 and in 1982.
- In 2026 the ratio climbed back to about 40, close to the 1999 record.
- With the slider, the model line rises slowly at first and then very fast: a P/E ratio of 40 needs
  $R-g = 1.25$ percent, against 3 percent for 16.7. Small changes in expected growth or in the interest
  rate can justify big changes in stock prices. That is why it is hard to tell a bubble from a change
  in the fundamentals.

#### Key numbers: averages, peaks and the growth rate the market expects

The textbook prints an average of 17.4. That is the average up to March 2023, the last month in the
textbook's figure. With the data up to today the average is a little higher.

In [33]:
book_sample = cape[cape.date <= "2023-03-01"]
print(f"Average P/E ratio, 1881 to March 2023: {book_sample.cape.mean():.1f}   (textbook: 17.4)")
print(f"Average P/E ratio, 1881 to {cape.date.iloc[-1]:%B %Y}: {cape.cape.mean():.1f}")
for year in [1929, 1999]:
    top = cape.loc[cape[cape.date.dt.year == year].cape.idxmax()]
    print(f"Peak of {year}: {top.cape:.1f} in {top.date:%B %Y}")

Average P/E ratio, 1881 to March 2023: 17.4   (textbook: 17.4)
Average P/E ratio, 1881 to October 2026: 17.8
Peak of 1929: 32.6 in September 1929
Peak of 1999: 44.2 in December 1999


Turn equation (10) around: for a given payout ratio, the P/E ratio tells us the value of $R-g$ that the
market's price implies, $R - g = \text{payout}/(P/E)$.

In [34]:
last = cape.iloc[-1]
for name, ratio in [("the long-run average", cape.cape.mean()), (f"{last.date:%B %Y}", last.cape)]:
    print(f"P/E ratio of {ratio:.1f} ({name}): R - g = {0.5 / ratio:.2%} with a payout ratio of 0.5")

P/E ratio of 17.8 (the long-run average): R - g = 2.81% with a payout ratio of 0.5
P/E ratio of 40.7 (October 2026): R - g = 1.23% with a payout ratio of 0.5


Today's prices are consistent with the simple model only if investors expect dividends to grow almost as
fast as the interest rate. Either they do, or prices are above their fundamental value.

**Your turn.** Exercise 8 of the textbook prices a stock that pays a dividend of 10 dollars. Use a `for`
loop and `stock_price` to compute its price for $R = 4$ percent and $g$ = 0, 2, 3 and 3.9 percent. What
happens to the price as $g$ gets close to $R$?

<details><summary>Hint</summary>

Loop over a list of growth rates: `for g in [0.0, 0.02, 0.03, 0.039]:` and print `stock_price(10, 0.04, g)`
inside the loop. With $g = 0$ the price is $10/0.04 = 250$ dollars.

</details>

In [35]:
# Your turn: try it here

## Investment in the data

Finally, we split U.S. private investment into its parts and see which ones move most in booms and recessions, in the United States and in other countries.

### Fig. 4: Components of private investment (as a share of GDP)

**Goal.** See which parts of investment move most in recessions.

In the national accounts, private investment has three parts:

- **nonresidential fixed investment**: equipment and structures bought by businesses, and intellectual
  property products such as research and software;
- **residential fixed investment**: new housing;
- **inventory investment**: the change in the stock of goods that firms have produced but not yet sold.

In [36]:
invest = pd.read_csv(DATA_URL + "fig17-4.csv")
recessions = pd.read_csv(DATA_URL + "recessions.csv", parse_dates=["peak", "trough"])
quarterly = pd.read_csv(DATA_URL + "investment_quarterly.csv", parse_dates=["date"])
invest.head()

,year,total,nonres_fixed,residential,inventories
0,1975,15.2681,11.6788,3.9606,-0.3713
1,1976,17.2532,11.7084,4.6312,0.9136
2,1977,19.0512,12.4439,5.5350,1.0723
3,1978,20.3426,13.3771,5.8688,1.0967
4,1979,20.5401,14.2286,5.6270,0.6845


Each row of `invest` is a year, and each column a share of GDP in percent; `total` is the sum of the
three parts. `recessions` lists the U.S. recessions dated by the NBER, and `quarterly` has the total and
residential shares for every quarter.

To draw all the parts with one `px.line` call, Plotly Express wants a **long** table: one row per year
and component. We first give the columns readable names, which then appear in the legend.

> **Python note.** `.rename(columns=...)` renames columns with a dictionary. `.melt(id_vars="year")`
> turns a wide table into a long one: the column names go into a new column (`component`) and the
> values into another (`share`). `pd.to_datetime(...)` turns the year into a date (1 January), so that
> the years line up with the recession dates.

In [37]:
NAMES = {"total": "Total", "nonres_fixed": "Nonresidential fixed", "residential": "Residential",
         "inventories": "Inventory"}
COLORS = {"Total": BLUE, "Nonresidential fixed": GREEN, "Residential": TEAL, "Inventory": "#5aa0d8"}
invest = invest.rename(columns=NAMES)
annual_long = invest.melt(id_vars="year", var_name="component", value_name="share")
annual_long["date"] = pd.to_datetime(annual_long.year.astype(str))

recent = quarterly[quarterly.date >= "1975-01-01"].rename(columns=NAMES)
quarterly_long = recent.melt(id_vars="date", var_name="component", value_name="share")
low = recent.loc[recent[recent.date.dt.year == 2009].Total.idxmin()]      # the lowest quarter of 2009
annual_long.head()

,year,component,share,date
0,1975,Total,15.2681,1975-01-01
1,1976,Total,17.2532,1976-01-01
2,1977,Total,19.0512,1977-01-01
3,1978,Total,20.3426,1978-01-01
4,1979,Total,20.5401,1979-01-01


`shade_recessions` shades the recessions. It adds one shaded rectangle (`add_vrect`) from the peak to the
trough of each recession.

> **Python note.** `zip(a, b)` hands over one pair of values at a time, here the peak and the trough of
> each recession.

In [38]:
def shade_recessions(fig, start):
    """Shade every NBER recession that ends after the date start."""
    for peak, trough in zip(recessions.peak, recessions.trough):
        if trough >= pd.Timestamp(start):
            fig.add_vrect(x0=peak, x1=trough, fillcolor=BLUE, opacity=0.12, line_width=0)
    return fig


def components_chart(frequency="annual"):
    """Fig. 17.4: investment and its components, percent of GDP, with recessions shaded."""
    title = "Investment and its components: all of them fall in recessions"
    data = annual_long
    if frequency == "quarterly":
        title = f"Quarterly data: total investment fell to {low.Total:.1f}% of GDP in {low.date.year}Q{low.date.quarter}"
        data = quarterly_long
    fig = px.line(data, x="date", y="share", color="component", color_discrete_map=COLORS, title=title,
                  labels={"date": "Year", "share": "Percent of GDP", "component": "Component"})
    shade_recessions(fig, "1975-01-01")
    fig.add_hline(y=0, line_color="black", line_width=1)
    fig.update_layout(xaxis_range=["1975-01-01", "2026-12-31"], yaxis=dict(range=[-5, 25], tickvals=[0, 5, 10, 15, 20, 25]),
                      legend=dict(orientation="h", x=0.01, y=0.99, title=None))
    return fig

> **Python note.** A `ToggleButtons` widget shows one button per option; the function receives the
> option you click.

In [39]:
interact(components_chart,
         frequency=ToggleButtons(options=["annual", "quarterly"], value="annual", description="data"));

In [40]:
# @title Website version: Fig. 4 with a precomputed slider for annual or quarterly data
slider_figure(components_chart, "frequency", ["annual", "quarterly"], start="annual", label="data")

**Sources.** Data: U.S. Bureau of Economic Analysis, NIPA Table 1.1.5, via FRED: [gross private domestic investment (`GPDIA`)](https://fred.stlouisfed.org/series/GPDIA), [private nonresidential fixed investment (`PNFIA`)](https://fred.stlouisfed.org/series/PNFIA), [private residential fixed investment (`PRFIA`)](https://fred.stlouisfed.org/series/PRFIA), [change in private inventories (`CBIA`)](https://fred.stlouisfed.org/series/CBIA) and [GDP (`GDPA`)](https://fred.stlouisfed.org/series/GDPA);
quarterly shares of GDP from NIPA Table 1.1.10, via FRED ([`A006RE1Q156NBEA`](https://fred.stlouisfed.org/series/A006RE1Q156NBEA), [`A011RE1Q156NBEA`](https://fred.stlouisfed.org/series/A011RE1Q156NBEA));
recessions: [NBER business cycle dates](https://www.nber.org/research/data/us-business-cycle-expansions-and-contractions), via FRED ([`USREC`](https://fred.stlouisfed.org/series/USREC)).
Prepared files: [`data/ch17/fig17-4.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/fig17-4.csv), [`data/ch17/investment_quarterly.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/investment_quarterly.csv), [`data/ch17/recessions.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/recessions.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/README.md)).
Chart: inspired by Fig. 17.4 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Total investment averaged 17.8 percent of GDP from 1975 to 2025, but it swings a lot: from 19.6 percent
  in 2006 to 13.3 percent in 2009, the lowest value in the figure.
- Residential investment peaked at 6.6 percent of GDP in 2005 and fell to 2.4 percent in 2011, less than
  half: the housing bust of the Great Recession.
- Inventory investment is small but **procyclical**: it was negative in the recession years 1975, 1980,
  1982, 1991, 2001, 2008, 2009 and 2020, and in 1983, just after a recession. Firms run down their
  inventories when sales fall.
- Click a name in the legend to hide or show that component. Click **quarterly**: the textbook's
  "less than 13 percent of GDP in 2009" is a quarterly number; the annual value is 13.3.

#### Beyond the United States: investment in 35 countries

Do housing and inventories swing as much in other countries? The OECD's national accounts let us build the same
three parts for 35 countries, from 1970 or 1995 to the latest year:

- **private investment** (`total`) is all fixed investment minus that of the government, plus inventory investment;
- **residential** investment is the investment in dwellings (it also includes a little public housing);
- **nonresidential fixed** investment is the rest of private fixed investment.

For the United States this gives the BEA's numbers of `fig17-4.csv` within 0.4 points of GDP in every year.

In [41]:
world = pd.read_csv(DATA_URL + "investment_components.csv")
world.head(3)

,iso3,country,year,total,nonres_fixed,residential,inventories,region
0,AUS,Australia,1970,27.73,21.59,4.69,1.45,Asia and Oceania
1,AUS,Australia,1971,26.07,20.73,4.99,0.35,Asia and Oceania
2,AUS,Australia,1972,24.04,19.38,5.34,-0.68,Asia and Oceania


The chart function draws one country at a time, with the same names and colors as Fig. 4. The vertical axis keeps
the textbook's scale unless a country invests more than 25 percent of GDP; then it switches to a wider fixed scale.

In [42]:
def components_by_country(country="United States"):
    """Fig. 17.4 for one country: private investment and its parts, percent of GDP (OECD, annual)."""
    rows = world[world.country == country].rename(columns=NAMES)
    long = rows.melt(id_vars=["iso3", "country", "year", "region"], var_name="component", value_name="share")
    low, high = rows.loc[rows.Total.idxmin()], rows.loc[rows.Total.idxmax()]
    top = 25
    if high.Total > 25:
        top = 55                                # a second, fixed scale for the high-investment countries
    fig = px.line(long, x="year", y="share", color="component", color_discrete_map=COLORS, hover_name="country",
                  labels={"year": "Year", "share": "Percent of GDP", "component": "Component"},
                  title=f"{country}: investment from {low.Total:.1f}% ({low.year}) to {high.Total:.1f}% "
                        f"({high.year}) of GDP")
    fig.add_hline(y=0, line_color="black", line_width=1)
    fig.update_layout(xaxis_range=[1969, 2026], yaxis=dict(range=[-5, top]),
                      legend=dict(orientation="h", x=0.01, y=0.99, title=None))
    return fig

The widget below is the Colab version. The website version needs one precomputed chart per country, joined by a
menu; the helper cell right after the widget builds it.

> **Python note.** A `Dropdown` is a menu from which you pick one entry. `sorted(world.country.unique())` lists every
> country once, in alphabetical order.

In [43]:
interact(components_by_country,
         country=Dropdown(options=sorted(world.country.unique()), value="United States", description="country",
                          style=WIDE));

In [44]:
# @title Helper for the website versions with a country menu (run this cell; you do not need to read it)
def dropdown_figure(make_figure, options, start):
    """Precompute make_figure(option) for every option and join the results with a dropdown menu.

    The website cannot run Python, so a figure with a country menu gets this precomputed version."""
    figures = [make_figure(o) for o in options]
    first = options.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which precomputed figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    buttons = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(), "yaxis.range": f.layout.yaxis.range,
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        buttons.append(dict(label=str(options[i]), method="update",
                            args=[{"visible": [o == i for o in owner]}, layout]))
    fig.update_layout(updatemenus=[dict(type="dropdown", active=first, buttons=buttons, x=0, y=1.02,
                                        xanchor="left", yanchor="bottom")],
                      margin=dict(t=110))
    return fig

In [45]:
# @title Website version: Fig. 4 for 35 countries, with a country menu
dropdown_figure(components_by_country, sorted(world.country.unique()), start="United States")

**Sources.** Data: OECD, National Accounts at a Glance, Chapter 3, investment (gross fixed capital formation), the government's share of it and investment in dwellings (dataflows `DSD_NAAG@DF_NAAG_III` and `DSD_NAAG@DF_NAAG_III_GFCF`); OECD national accounts Table 1, GDP and changes in inventories (`DSD_NAMAIN10@DF_TABLE1_EXPENDITURE`); all via the [OECD Data Explorer](https://data-explorer.oecd.org/).
Prepared file: [`data/ch17/investment_components.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/investment_components.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch17/README.md)).
Chart: inspired by Fig. 17.4 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

Pick a country in the menu above the chart.

**What to notice.**

- Housing booms were not only American. Choose Ireland: its residential investment reached 13.5 percent of GDP in
  2006, twice the U.S. peak, and fell to 2.0 percent in 2011. Greece's reached 11.0 percent in 2007.
- Germany had no housing boom in the 2000s. Its total still fell, to 15.1 percent of GDP in 2009, and inventories
  fell by 1.1 percent of GDP that year, as in Japan (−1.0). The recession of 2009 was worldwide.
- Look at your data: Ireland's investment jumps to 51.5 percent of GDP in 2019. Multinational firms moved their
  patents and other intellectual property to Ireland, which counts as investment there. It is accounting, not new
  factories.

#### Key numbers: which part of investment falls in a recession?

For four recessions we compute how much each part of investment changed, in percentage points of GDP,
from the year before the recession to its worst year.

In [46]:
table = invest.set_index("year")
rows = []
for start, end in [(1979, 1982), (2000, 2002), (2006, 2009), (2019, 2020)]:
    change = table.loc[end] - table.loc[start]
    change["episode"] = f"{start} to {end}"
    rows.append(change)
pd.DataFrame(rows).set_index("episode").round(2)

,Total,Nonresidential fixed,Residential,Inventory
episode,,,,
1979 to 1982,-3.16,0.28,-2.31,-1.13
2000 to 2002,-2.22,-2.24,0.37,-0.35
2006 to 2009,-6.22,-1.31,-3.37,-1.54
2019 to 2020,-0.47,-0.46,0.47,-0.48


Every recession is different. Housing collapsed in the early 1980s, when interest rates were very high,
and again in 2006–2009, when it explains more than half of the fall of 6.2 points. The dot-com bust of
2000–2002 hit business investment, while housing even grew. In 2020 the recession lasted only two months,
so annual data hide most of it.

#### Key numbers: house prices and leverage

The arbitrage equation also prices a house. An investor can keep the down payment in the bank, or buy a
condominium with a mortgage, rent it out for a year and sell it. Equating the two returns gives the house price of equation (13):

$$p^{\text{house}} = \frac{\text{rent}}{\tau\bar{x}R + (1-\tau)R + \bar{d} - \Delta p^{\text{house}}/p^{\text{house}}}$$

where $\bar{x}$ is the down payment as a share of the price and $\tau$ the tax rate at which mortgage
interest can be deducted. We use illustrative numbers: a rent of 6,000 dollars a year, $R = 4$ percent,
$\bar{d} = 2$ percent, $\tau = 0.3$ and expected price growth of 2 percent.

In [47]:
def house_price(rent, R, d, gp, tau, x):
    """Price of a house: rent / (tau x R + (1 - tau) R + d - gp)   (equation 13; gp is the expected price growth)."""
    return rent / (tau * x * R + (1 - tau) * R + d - gp)


base = house_price(6000, 0.04, 0.02, 0.02, 0.3, 0.2)
print(f"Baseline price (20% down payment, 2% price growth): {base:,.0f} dollars")
print(f"Down payment falls to 10%:          {house_price(6000, 0.04, 0.02, 0.02, 0.3, 0.1) / base - 1:+.1%}")
print(f"Expected price growth rises to 3%:  {house_price(6000, 0.04, 0.02, 0.03, 0.3, 0.2) / base - 1:+.1%}")

Baseline price (20% down payment, 2% price growth): 197,368 dollars
Down payment falls to 10%:          +4.1%
Expected price growth rises to 3%:  +49.0%


With these numbers, cheaper credit (a smaller down payment) raises the price by about 4 percent, but a
belief that prices will grow 1 point faster raises it by about half. Expected capital gains can feed on
themselves, which is how a housing bubble grows.

**Your turn.** The textbook puts the average of private investment over the last few decades at about 17
percent of GDP. Compute the average of the `Total` column of `invest` since 2000. Then compute the share of
nonresidential fixed investment in total investment in 2025. Finally, use `world` to find the country with the
highest average private investment over all its years. Is the U.S. average high or low among the 35 countries?

<details><summary>Hint</summary>

Filter the years with `invest[invest.year >= 2000]` and use `.Total.mean()`. For the share, take the 2025
row, `invest[invest.year == 2025]`, and divide its `"Nonresidential fixed"` column by its `Total` column. For the
countries, `world.groupby("country").total.mean()` averages each country's rows; add `.sort_values()` to rank them.

</details>

In [48]:
# Your turn: try it here

## Concluding remarks

- The **arbitrage equation**, equation (1), says that two equally safe investments must earn the same return.
  Applied to physical capital, it says that a firm keeps adding capital as long as its marginal product is above
  the **user cost of capital**, $R + \bar{d} - \Delta p_k/p_k$ (equation 2), and stops where the two are equal.
- A corporate income tax divides the user cost by $1-\tau$ (equation 3).
- With diminishing returns, equations (4) and (5) turn the user cost into a **desired capital stock** proportional to
  $uc^{-3/2}$, and equations (6) and (7) into a capital-output ratio $Y/K = 3\cdot uc$ and an investment rate
  $(g_K + \bar{d})/(3\cdot uc)$. A 35 percent tax cuts desired capital by about half and the investment rate by 35 percent.
- Applied to stocks, the same equation, (8), gives $p_s = \text{dividend}/(R-g)$ (equation 9): a stock is worth the present
  value of its dividends. The P/E ratio of equation (10) then depends on $R-g$, which makes it very sensitive to
  expected growth and hard to use for spotting bubbles.
- **Tobin's $q$**, equations (11) and (12), links the stock market to investment: $q > 1$ exactly when the marginal
  product of capital is above the user cost. The house price of equation (13) shows how leverage, the mortgage tax
  deduction and, above all, expected price gains move house prices.
- In the data, total investment and especially housing and inventories fall sharply in recessions.
- The next chapter turns to **the government and the macroeconomy**: how governments spend, tax and
  borrow, and what their budget constraint implies for debt and deficits.

## Practice problems

These problems put the chapter to work in three ways: **conceptual problems** apply the arbitrage equation to real cases, **data-based problems** send you back to the chapter's data, and **modeling problems** use the interactive models to answer policy questions. Each problem is marked Basic, Intermediate or Challenge: try it on your own first, then open the Hint, and only then the Solution. The sliders of the modeling problems run in Colab (open the notebook with the badge at the top); on the website, use the website version of each figure above.

### Conceptual problems

**Problem 1 (Basic): A minimum tax for Ireland's multinationals.** Ireland taxes company profits at 12.5 percent, one of the lowest rates in the tax table after Fig. 2. In October 2021 most members of the OECD/G20 Inclusive Framework agreed that multinational groups with a revenue of at least 750 million euros should pay a tax rate of at least 15 percent ([OECD statement](https://www.oecd.org/content/dam/oecd/en/topics/policy-issues/beps/statement-on-a-two-pillar-solution-to-address-the-tax-challenges-arising-from-the-digitalisation-of-the-economy-october-2021.pdf)), and Ireland now charges such groups a top-up tax that brings them to that rate ([Revenue, the Irish tax agency](https://www.revenue.ie/en/companies-and-charities/pillar-two/what-is/pillar-two-rules.aspx)). Keep the user cost before taxes of 7.9 percent that the tax table uses.

(a) Use equation (3) to compute the user cost of capital of a large multinational in Ireland, first with a tax rate of 12.5 percent and then with 15 percent.

(b) By what percentage does its desired capital stock change? Use equation (5).

(c) By what percentage does its investment rate, equation (7), change? Is the minimum tax a big deal for the machines and buildings that multinationals own in Ireland?

<details><summary>Hint</summary>

The user cost is $0.079/(1-\tau)$. Desired capital is proportional to $uc^{-3/2}$ and the investment rate to $1/uc$, so for (b) and (c) you only need the ratio of the two user costs, $(1-0.125)/(1-0.15)$.

</details>

**Your answer:**

<details><summary>Solution</summary>

**(a)** Equation (3) divides the user cost before taxes by the share of income the firm keeps, $1-\tau$:

$$uc_{12.5} = \frac{0.079}{1-0.125} = \frac{0.079}{0.875} \approx 0.0903, \qquad uc_{15} = \frac{0.079}{1-0.15} = \frac{0.079}{0.85} \approx 0.0929.$$

The user cost rises from 9.03 to 9.29 percent, by the factor $0.875/0.85 \approx 1.029$, that is by 2.9 percent.

**(b)** Equation (5) makes desired capital proportional to $uc^{-3/2}$, and nothing else in it changes:

$$\frac{K_{15}}{K_{12.5}} = \left(\frac{uc_{12.5}}{uc_{15}}\right)^{3/2} = \left(\frac{0.85}{0.875}\right)^{3/2} \approx 0.957.$$

Desired capital falls by about 4.3 percent. The elasticity of $-1.5$ gives almost the same answer: $-1.5 \times 2.9 \approx -4.4$ percent.

**(c)** Equation (7) makes the investment rate proportional to $1/uc$, so it falls by the factor $0.85/0.875 \approx 0.971$, or by 2.9 percent: an investment rate of 20 percent of output would become about 19.4 percent.

These are small effects. A rise of 2.5 points in the tax rate changes the share the firm keeps only from 0.875 to 0.85. The minimum tax matters much more for where multinationals *report* their profits than for where they build factories: recall from Fig. 4 that Ireland's measured investment jumped in 2019 when firms moved their intellectual property there, an accounting move rather than new machines.

</details>

**Problem 2 (Intermediate): Why did U.S. house prices hold up in 2022–23?** Real interest rates in the United States rose fast after 2021. The yield on 10-year inflation-indexed Treasury bonds, a measure of the real interest rate, averaged −0.91 percent in 2021 and 1.68 percent in 2023 ([FRED, `DFII10`](https://fred.stlouisfed.org/series/DFII10)), a rise of about 2.6 points. Yet real house prices did not fall: the S&P CoreLogic Case-Shiller U.S. National Home Price Index ([FRED, `CSUSHPISA`](https://fred.stlouisfed.org/series/CSUSHPISA)), divided by the consumer price index ([FRED, `CPIAUCSL`](https://fred.stlouisfed.org/series/CPIAUCSL)), was 4.6 percent higher in 2023 than in 2021 (our calculation from the annual averages).

(a) Start from the house of the Key numbers after Fig. 4: a rent of 6,000 dollars a year, $R = 0.04$, $\bar{d} = 0.02$, $\tau = 0.3$, $\bar{x} = 0.2$ and expected price growth of 2 percent. Raise $R$ by 2.6 points. What price does equation (13) predict, and how large is the fall?

(b) By how many points would expected price growth, $\Delta P^{\text{house}}/P^{\text{house}}$, have had to rise to keep the price unchanged?

(c) In the model, every buyer borrows at today's rate $R$. In practice, many U.S. owners hold 30-year fixed-rate mortgages taken out when rates were low: the average 30-year fixed rate was 2.96 percent in 2021 and 6.81 percent in 2023 ([Freddie Mac, via FRED, `MORTGAGE30US`](https://fred.stlouisfed.org/series/MORTGAGE30US)). Why might this keep house prices up when rates rise?

<details><summary>Hint</summary>

(a) Only the interest terms of the denominator change, by $(\tau\bar{x} + 1 - \tau)\,\Delta R$ (see the table after equation (13)). (b) Expected price growth enters the denominator with a minus sign and a coefficient of one. (c) Would you sell your house if moving meant swapping a 3 percent loan for a 7 percent one?

</details>

**Your answer:**

<details><summary>Solution</summary>

**(a)** At the start, the denominator of equation (13) is

$$\tau\bar{x}R + (1-\tau)R + \bar{d} - \frac{\Delta P^{\text{house}}}{P^{\text{house}}} = 0.3\times0.2\times0.04 + 0.7\times0.04 + 0.02 - 0.02 = 0.0024 + 0.028 = 0.0304,$$

so the price is $6{,}000/0.0304 \approx 197{,}368$ dollars. With $R = 0.066$ the denominator becomes

$$0.3\times0.2\times0.066 + 0.7\times0.066 + 0.02 - 0.02 = 0.00396 + 0.0462 = 0.05016,$$

and the price $6{,}000/0.05016 \approx 119{,}617$ dollars, a fall of 39.4 percent. A shortcut: the interest terms add up to $(\tau\bar{x} + 1 - \tau)R = 0.76\,R$, so the denominator rises by $0.76 \times 0.026 = 0.01976$. Because the denominator is a small number, this adds 65 percent to it. The model predicts a housing crash.

**(b)** To keep the price, the denominator must stay at 0.0304. Expected price growth enters it with a coefficient of $-1$, so it must rise by as much as the interest terms did: $0.76 \times 0.026 \approx 0.020$, about 2 points, from 2 to 4 percent a year after inflation. To match the 4.6 percent rise in real prices, the denominator must fall to $0.0304/1.046 \approx 0.0291$, which needs expected growth of 4.1 percent, 2.1 points more than before. So the model fits the facts only if buyers became much more optimistic about house prices at the very moment interest rates jumped. That is possible, but it is a strong assumption.

**(c)** An owner with a 3 percent loan who sells her house must repay that loan, and pay about 7 percent on the loan for her next home. Many owners therefore stayed where they were, and few houses came on the market. Fewer sellers support the price, which the model leaves out, because it assumes that anyone can buy or sell at no cost. [Batzer, Coste, Doerner and Seiler (2024)](https://www.fhfa.gov/research/papers/wp2403), economists at the Federal Housing Finance Agency, estimate that this "lock-in" prevented about 1.33 million home sales between the second quarter of 2022 and the end of 2023, and that the smaller supply raised prices by 5.7 percent, more than the 3.3 percent by which the higher rates lowered them directly. The arbitrage equation still pulls prices toward equation (13), but when buying and selling are costly, it can take years.

</details>

### Data-based problems

**Problem 3 (Intermediate): The race to the bottom in corporate taxes.** Many governments cut their corporate tax rates after 2000. Check this with `corporate_tax.csv`, which holds the statutory tax rates of the central government from the OECD's Corporate Tax Statistics ([OECD Data Explorer](https://data-explorer.oecd.org/)), for the 33 jurisdictions of the tax table after Fig. 2. Use 2025 as the last year, because one jurisdiction has no rate for 2026 yet.

(a) What was the average rate of the 33 jurisdictions in 2000, and in 2025? How many of them cut their rate, kept it, or raised it?

(b) Which three jurisdictions made the biggest cuts?

(c) Keep the user cost before taxes fixed. Equations (3) and (5) then say that desired capital is proportional to $(1-\tau)^{3/2}$. By what factor did the change in its tax rate raise each jurisdiction's desired capital stock between 2000 and 2025? Report the median factor and the three largest ones.

<details><summary>Hint</summary>

`rates.pivot(index="country", columns="year", values="cit_rate")` turns the long table into a wide one, with one row per country and one column per year. Then `rate_change = wide[2025] - wide[2000]` is a column of changes, and `(rate_change < 0).sum()` counts the cuts, because Python counts each `True` as 1. The rates are in percent: divide them by 100 before you use them in $(1-\tau)^{3/2}$.

</details>

In [49]:
# Your answer here

<details><summary>Solution</summary>

We read the file, give each year its own column with `pivot`, and compare the columns for 2000 and 2025.

```python
rates = pd.read_csv(DATA_URL + "corporate_tax.csv")
wide = rates.pivot(index="country", columns="year", values="cit_rate")    # one row per country
rate_change = wide[2025] - wide[2000]
print(f"Average rate: {wide[2000].mean():.1f}% in 2000, {wide[2025].mean():.1f}% in 2025")
print(f"Cut: {(rate_change < 0).sum()}, unchanged: {(rate_change == 0).sum()}, raised: {(rate_change > 0).sum()}")
print("Raised:", list(rate_change[rate_change > 0].index))
print(f"Standard deviation: {wide[2000].std():.1f} points in 2000, {wide[2025].std():.1f} in 2025")
print(rate_change.sort_values().head(3).round(1))
```

The average rate fell from 30.7 percent in 2000 to 23.8 percent in 2025. Twenty-five jurisdictions cut their rate, six kept it and only two raised it, Chile and Hong Kong. The biggest cuts were made by India (27.2 points), Germany (26.4) and the Democratic Republic of the Congo (20.0).

For (c), desired capital is proportional to $(1-\tau)^{3/2}$, so the factor is $\left((1-\tau_{2025})/(1-\tau_{2000})\right)^{3/2}$:

```python
factor = ((1 - wide[2025] / 100) / (1 - wide[2000] / 100)) ** 1.5      # equations (3) and (5)
print(f"Median factor: {factor.median():.2f}")
print(factor.sort_values(ascending=False).head(3).round(2))
```

The median factor is 1.12: for the typical jurisdiction, the tax cuts alone raised desired capital by about 12 percent. India's factor is 1.97, Germany's 1.76 and Congo's 1.66, so in the model their tax cuts almost doubled the capital stock that firms want. Chile, with a factor of 0.80, went the other way.

Two cautions. First, the file holds the central government's rate only, so taxes that regions or cities charge on profits are left out (see the data notes). Second, the factors assume that nothing else in the tax code changed, such as the deductions that firms may claim. Still, the pattern is clear: rates fell almost everywhere, and the spread between them narrowed (the standard deviation fell from 10.1 to 7.5 points), which is what you would expect if countries compete for capital.

</details>

**Problem 4 (Challenge): Does housing warn of recessions?** Edward Leamer argued that, of all the parts of GDP, residential investment gives the best early warning of a U.S. recession ([Leamer 2007, NBER Working Paper 13428](https://www.nber.org/papers/w13428)). Test his claim with the quarterly shares of GDP in `investment_quarterly.csv` ([BEA, NIPA Table 1.1.10, via FRED: residential `A011RE1Q156NBEA`](https://fred.stlouisfed.org/series/A011RE1Q156NBEA) and [total `A006RE1Q156NBEA`](https://fred.stlouisfed.org/series/A006RE1Q156NBEA)) and the recession dates in `recessions.csv` ([NBER business cycle dates](https://www.nber.org/research/data/us-business-cycle-expansions-and-contractions)).

(a) For each of the 12 recessions since 1948, compute the change in the residential share of GDP over the year before the recession, from four quarters before the peak to the quarter that contains the peak. Do the same for the share of total investment. Before how many recessions did each share fall?

(b) Leamer noted that housing gave no warning of the recessions that began in 1953 and 2001. Do our numbers agree?

(c) Compute the same four-quarter change for the first quarter of 2023. The NBER list has no recession after 2020. Was this a false alarm? Use equation (13) and Problem 2 to explain why housing fell first, and what else would be needed for a recession.

<details><summary>Hint</summary>

Loop over `dates.peak`. For each peak, `housing[housing.date <= peak]` keeps the quarters up to the one that contains the peak; `.iloc[-1]` is that last quarter and `.iloc[-5]` the quarter one year earlier. Round each change to one decimal before you test whether it is below zero.

</details>

In [50]:
# Your answer here

<details><summary>Solution</summary>

For each peak we take the last quarter that starts on or before the peak month, and the quarter four rows earlier, and subtract their shares.

```python
housing = pd.read_csv(DATA_URL + "investment_quarterly.csv", parse_dates=["date"])
dates = pd.read_csv(DATA_URL + "recessions.csv", parse_dates=["peak", "trough"])
housing_falls, total_falls = 0, 0
for peak in dates.peak:
    before = housing[housing.date <= peak]         # every quarter up to the one with the peak
    now, year_ago = before.iloc[-1], before.iloc[-5]
    res = round(now.residential - year_ago.residential, 1) + 0.0
    tot = round(now.total - year_ago.total, 1) + 0.0
    if res < 0:
        housing_falls += 1
    if tot < 0:
        total_falls += 1
    print(f"Peak {peak:%b %Y}: residential {res:+.1f}, total {tot:+.1f} points of GDP")
print(f"Falls in the year before a peak: residential {housing_falls} of 12, total {total_falls} of 12")
```

The residential share fell in the year before 11 of the 12 recessions; the exception is 2020, when the recession came with the COVID-19 pandemic ([NBER announcement](https://www.nber.org/news/business-cycle-dating-committee-announcement-june-8-2020)). The share of total investment fell before only 8 of them, and it even rose before the recessions of 1948, 1953, 1973 and 1981. Housing is the better warning, as Leamer said. The falls are often half a point or more: 0.6 before 1957, 0.7 before 1960, 1973 and 1980, and 1.4 before December 2007, at the start of the Great Recession.

**(b)** Yes. Before July 1953 and March 2001 the residential share fell by only 0.1 point, too little to count as a warning. Leamer traces those two recessions to other causes, the cut in defense spending after the Korean armistice and the end of the internet boom; in 2000–2002 it was business investment that collapsed (see the Key numbers after Fig. 4).

**(c)**

```python
q2022 = housing[housing.date == "2022-01-01"].residential.iloc[0]
q2023 = housing[housing.date == "2023-01-01"].residential.iloc[0]
print(f"Residential share: {q2022:.1f}% in 2022Q1, {q2023:.1f}% in 2023Q1, change {q2023 - q2022:+.1f} points")
rest_2022 = housing[housing.date == "2022-01-01"].total.iloc[0] - q2022
rest_2023 = housing[housing.date == "2023-01-01"].total.iloc[0] - q2023
print(f"Rest of private investment: {rest_2022:.1f}% -> {rest_2023:.1f}% of GDP")
```

The residential share fell from 4.8 to 4.0 percent of GDP, a drop of 0.8 points, as large as before most recessions, and no recession followed in our data. It was a false alarm. Housing fell first because its user cost, the denominator of equation (13), reacts most to the interest rate: Problem 2 showed that 2.6 more points of real interest rate raise it by 65 percent. In Leamer's account, a recession follows when the slump in housing spreads to the rest of the economy, starting with spending on consumer durables. In 2022–23 the rest of private investment (total minus residential) fell only from 14.2 to 13.9 percent of GDP. A single indicator can warn, but it cannot prove that a recession is coming.

</details>

### Modeling problems

**Problem 5 (Basic): Buy machines before the tariff?** Firms often rush to buy before a price rise. In early 2025 the staff of the Federal Reserve linked a surge in U.S. imports to purchases made ahead of expected tariff increases ([FOMC minutes, May 2025](https://www.federalreserve.gov/monetarypolicy/fomcminutes20250507.htm)), and U.S. equipment investment grew at an annual rate of 22.5 percent in the first quarter, in the BEA's first estimate ([BEA, Survey of Current Business, May 2025](https://apps.bea.gov/scb/issues/2025/05-may/0525-gdp-economy.htm)). Take a hypothetical firm in Fig. 1's model, with $R = 0.02$, $\bar{d} = 0.08$ and $\bar{A} = 0.3$, which holds its desired capital stock, $K_0 = 1$. A tariff announced for next year makes the firm expect the price of its machines to rise by 3 percent during this year.

(a) Move the slider for the expected price rise from 0 to 0.03. What happens to the user cost, the desired capital stock and this year's investment?

(b) Once the tariff is in place, machine prices stop rising. Set the price rise back to 0, and the starting capital to the capital the firm owns after (a). What does the firm invest in the following year?

<details><summary>Hint</summary>

All three numbers are in the title of the chart. An expected rise in the price of a machine is a capital gain, so it enters the user cost of equation (2) with a minus sign. In (b), the firm's capital is the desired capital you read in (a), rounded to the slider's step of 0.1.

</details>

In [51]:
interact(desired_capital_diagram,
         R=fixed(0.02), d=fixed(0.08), A=fixed(A_BAR),
         gpk=FloatSlider(value=0.0, min=0.0, max=0.04, step=0.01, description="expected price rise Δp/p", style=WIDE),
         K0=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description="starting capital K0", style=WIDE));

**Source.** The model of Fig. 1 above, adapted from Fig. 17.1 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

In [52]:
# Your answer here

<details><summary>Solution</summary>

**(a)** The title changes from "User cost 10.0% | desired capital 1.00 | investment from K0 = 1.0: +0.08" to "User cost 7.0% | desired capital 1.71 | investment from K0 = 1.0: +0.79". A machine bought today can be sold dearer at the end of the year, so owning it costs 3 points less, and the user cost falls from $0.02 + 0.08 = 0.10$ to $0.02 + 0.08 - 0.03 = 0.07$. Equation (5) turns this 30 percent cut in the user cost into a 71 percent rise in desired capital, $(0.10/0.07)^{3/2} \approx 1.71$. Investment, which normally only replaces the 8 percent of capital that wears out each year, becomes almost ten times larger: $1.71 - 0.92 \times 1 \approx 0.79$.

**(b)** With $\Delta p/p = 0$ and $K_0 = 1.7$ the title shows investment of $-0.56$ (with the exact 1.71, $1 - 0.92 \times 1.71 \approx -0.57$). The firm now owns more capital than it wants, so it buys no machines at all and lets depreciation shrink its capital. You can check the numbers in the answer cell:

```python
uc_gain = user_cost(0.02, 0.08, gpk=0.03)
K_boom = desired_capital(uc_gain)
print(f"This year: user cost {uc_gain:.2f}, desired capital {K_boom:.3f}, investment {K_boom - 0.92 * 1.0:.3f}")
print(f"Next year: investment {desired_capital(user_cost(0.02, 0.08)) - 0.92 * K_boom:.3f}")
```

It prints a user cost of 0.07, desired capital of 1.707 and investment of 0.787 this year, and $-0.571$ the year after. With no new machines, capital shrinks by 8 percent a year, so it takes 7 years to fall back from 1.71 to 1. (Strictly, the machines also stay 3 percent dearer after the tariff, which raises the user cost to 0.103 and lowers desired capital a little further, to about 0.96.)

**The policy lesson.** An announced price rise does not create investment out of nothing: it moves investment forward in time, a boom this year and a bust after it. Real firms face costs of installing machines quickly, so the boom is spread out and smaller than in our model, but its direction is the same. The same logic applies to a temporary investment subsidy: when firms know that it ends next year, they bring their purchases forward.

</details>

**Problem 6 (Intermediate): Can the central bank cool the stock market?** In our data, which end in October 2026, Shiller's P/E ratio is 40.7, more than twice its long-run average of 17.8 (see the title of Fig. 3). Could higher interest rates bring stock prices back down? Use the sliders below, with the payout ratio fixed at one half.

(a) Set $R = 4$ percent. Which dividend growth rate $g$ on the slider brings the model's P/E ratio closest to the ratio of October 2026?

(b) The central bank now raises the real interest rate by one point, to $R = 5$ percent, and expected growth does not change. What P/E ratio does the model predict, and by what percentage would stock prices fall?

(c) Repeat (b) for a market at its long-run average: with a payout ratio of one half, a P/E ratio of 17.8 implies $R - g = 0.5/17.8$. By what percentage would prices fall after the same one-point rise? Why is the answer so different?

(d) Higher interest rates also reach investment. Take the firm of Fig. 1, with $R = 2$ percent, which holds its desired capital $K = 1$. Use `tobin_q` to compute its $q$ after a rise of the interest rate to 3 percent, and `desired_capital` with `user_cost` to find its new desired capital. What will the firm do?

<details><summary>Hint</summary>

By equation (10), the P/E ratio is $0.5/(R-g)$, so a P/E ratio of 40.7 needs $R - g = 0.5/40.7$. In (b) and (c), prices fall by the factor $(R-g)_{\text{before}}/(R-g)_{\text{after}}$. In (d), use `tobin_q(1.0, R=0.03)` and `desired_capital(user_cost(0.03, 0.08))`.

</details>

In [53]:
interact(pe_chart,
         R=FloatSlider(value=0.04, min=0.03, max=0.06, step=0.005, description="interest rate R", style=WIDE, readout_format=".3f"),
         g=FloatSlider(value=0.02, min=0.0, max=0.035, step=0.0025, description="dividend growth g", style=WIDE, readout_format=".4f"),
         payout=fixed(0.5));

**Source.** The chart of Fig. 3 above: data from [Robert J. Shiller, Online Data](https://shillerdata.com/); the model line of equation (10) is adapted from the discussion of Fig. 17.3 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

In [54]:
# Your answer here

<details><summary>Solution</summary>

**(a)** The P/E ratio of 40.7 needs $R - g = 0.5/40.7 \approx 1.23$ percent, so $g \approx 2.77$ percent. On the slider, $g = 0.0275$ is closest: the title shows a model ratio of $0.5/(0.04 - 0.0275) = 0.5/0.0125 = 40.0$.

**(b)** With $R = 0.05$ the gap widens to $R - g = 0.05 - 0.0275 = 0.0225$, and the model ratio falls to $0.5/0.0225 \approx 22.2$. Since the price is proportional to $1/(R-g)$, it falls by the factor $0.0125/0.0225 \approx 0.556$: stock prices would fall by 44.4 percent.

**(c)** At the long-run average, $R - g = 0.5/17.8 \approx 2.81$ percent. One more point gives 3.81 percent and a P/E ratio of $0.5/0.0381 \approx 13.1$, so prices fall by the factor $0.0281/0.0381$, that is by 26.3 percent. The same one-point rise does much less harm, because the price has an elasticity of $-1$ with respect to the gap $R-g$: one point is 80 percent of a gap of 1.25 points, but only 36 percent of a gap of 2.81 points. A market priced on a thin gap between interest rates and expected growth is very sensitive to interest rates.

**(d)**

```python
print(f"P/E ratio: {pe_ratio(0.5, 0.04, 0.0275):.1f} -> {pe_ratio(0.5, 0.05, 0.0275):.1f}")
print(f"Tobin's q at R = 3%: {tobin_q(1.0, R=0.03):.3f}")
print(f"New desired capital: {desired_capital(user_cost(0.03, 0.08)):.3f}")
```

It prints 40.0 → 22.2, a $q$ of 0.667 and a new desired capital stock of 0.867. With $q$ below 1 the firm is worth less than its machines: it stops investing until depreciation has brought its capital down by about 13 percent, which takes two years at 8 percent a year. (Parts (a)–(c) use the stock example of the Model overview, with $R$ around 4 to 5 percent; part (d) uses the machine baseline of Fig. 1, with $R = 2$ percent.)

**The policy answer.** In the model, a central bank that raises real interest rates can bring stock prices down a lot, and through $q$ it also lowers investment, which is one way monetary policy cools the economy. But the size of the fall depends on how thin the gap $R - g$ is, which nobody observes directly. A central bank that tries to target stock prices could easily do far more, or far less, than it intends.

</details>

**Problem 7 (Challenge): A Laffer curve for the corporate tax?** Does a higher corporate tax always raise more revenue? Take Fig. 2's economy: firms hold their desired capital stock, and the interest rate $R = 0.02$ is set outside the country, for example on world capital markets. The government collects $\tau\cdot MPK\cdot K$, and the title of the chart below adds output and this tax revenue.

(a) Move the slider from 5 to 70 percent. At which tax rate on the slider is the revenue highest?

(b) Show that revenue is $T = \frac{\tau}{3}\,Y$ and that output at the desired capital stock is $Y = \bar{A}L\left(\frac{\bar{A}(1-\tau)}{3\,uc_0}\right)^{1/2}$, where $uc_0 = R + \bar{d}$ is the user cost without the tax. Then derive the tax rate that maximizes revenue.

(c) Workers earn the other two thirds of output. At $\tau = 0.35$, compare the tax revenue with the fall in wages, relative to an economy with no corporate tax. Who pays the corporate tax in this model?

(d) The rates of our tax table run from 0 to about 36 percent in 2025, far below your answer to (b). Give two reasons why.

<details><summary>Hint</summary>

(b) Equation (4) says $MPK\cdot K = \frac{1}{3}Y$. Put $K^{\text{desired}}$ from equation (5), with $uc = uc_0/(1-\tau)$, into $Y = \bar{A}K^{1/3}L^{2/3}$. With the baseline, $\bar{A}/(3\,uc_0) = 1$, so revenue is proportional to $\tau(1-\tau)^{1/2}$: set its derivative to zero. (c) Wages are $\frac{2}{3}Y$; compare them with and without the tax.

</details>

In [55]:
def tax_revenue_diagram(tau=0.35):
    """Fig. 2 with output and the tax revenue tau x MPK x K at the desired capital stock in the title."""
    fig = corporate_tax_diagram(tau, R=0.02, d=0.08)
    K1 = desired_capital(user_cost(0.02, 0.08, tau=tau))
    Y1 = A_BAR * K1 ** (1 / 3) * L_BAR ** (2 / 3)
    revenue = tau * mpk(K1) * K1
    fig.update_layout(title=f"τ = {tau:.0%}  |  desired capital {K1:.2f}  |  output {Y1:.3f}  |  tax revenue {revenue:.4f}")
    return fig


interact(tax_revenue_diagram,
         tau=FloatSlider(value=0.35, min=0.05, max=0.7, step=0.05, description="corporate tax rate τ", style=WIDE));

**Source.** The model of Fig. 2 above, with a tax-revenue readout of our own, adapted from Fig. 17.2 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

In [56]:
# Your answer here

<details><summary>Solution</summary>

**(a)** Revenue rises from 0.0049 at 5 percent to 0.0282 at 35 percent and 0.0385 at 65 percent, then falls to 0.0383 at 70 percent. On the slider, 65 percent raises the most.

**(b)** At the desired capital stock, equation (4) gives $MPK = \frac{1}{3}Y/K$, so revenue is

$$T = \tau\cdot MPK\cdot K = \frac{\tau}{3}\,Y.$$

From equation (5), $K = L\left(\bar{A}/(3\,uc)\right)^{3/2}$, so $K^{1/3} = L^{1/3}\left(\bar{A}/(3\,uc)\right)^{1/2}$ and

$$Y = \bar{A}K^{1/3}L^{2/3} = \bar{A}L\left(\frac{\bar{A}}{3\,uc}\right)^{1/2} = \bar{A}L\left(\frac{\bar{A}(1-\tau)}{3\,uc_0}\right)^{1/2},$$

using $uc = uc_0/(1-\tau)$ from equation (3). With the baseline, $\bar{A} = 0.3$, $L = 1$ and $uc_0 = 0.10$, the ratio $\bar{A}/(3\,uc_0)$ is 1, so $Y = 0.3\,(1-\tau)^{1/2}$ and $T = 0.1\,\tau(1-\tau)^{1/2}$. The tax raises revenue on each unit of output but shrinks output. Setting the derivative to zero,

$$\frac{dT}{d\tau} = 0.1\left[(1-\tau)^{1/2} - \frac{\tau}{2}(1-\tau)^{-1/2}\right] = 0 \quad\Longrightarrow\quad 2(1-\tau) = \tau \quad\Longrightarrow\quad \tau^* = \frac{2}{3},$$

after multiplying the bracket by $2(1-\tau)^{1/2}$. At $\tau^* = 2/3$, revenue is $0.1\times\frac{2}{3}\times(1/3)^{1/2} \approx 0.0385$, and output is $0.3\times(1/3)^{1/2} \approx 0.173$, 42 percent below its level with no tax. Below 2/3, where every rate of our tax table lies, a tax cut loses revenue: it does not pay for itself. Only above 2/3 would a cut raise revenue.

**(c)** With no tax, output is 0.3 and wages are $\frac{2}{3}\times0.3 = 0.2$. At $\tau = 0.35$, output is $0.3\times0.65^{1/2} \approx 0.2419$, so wages fall to about 0.1612, a loss of 0.0388, while the government collects only 0.0282. Workers lose 1.37 dollars of wages for each dollar of revenue.

```python
Y_no_tax = A_BAR * desired_capital(user_cost(0.02, 0.08)) ** (1 / 3)
K35 = desired_capital(user_cost(0.02, 0.08, tau=0.35))
Y35 = A_BAR * K35 ** (1 / 3)
wage_loss = 2 / 3 * (Y_no_tax - Y35)
revenue_35 = 0.35 * mpk(K35) * K35
print(f"Output {Y_no_tax:.4f} -> {Y35:.4f}, wages fall by {wage_loss:.4f}, revenue {revenue_35:.4f}")
print(f"Wages lost per dollar of revenue: {wage_loss / revenue_35:.2f}")
```

In this model, the workers pay the corporate tax, and more than all of it. Because $R$ is fixed, the owners of capital still earn $R$ after tax: they simply invest less. With less capital per worker, the marginal product of labor and the wage fall.

**(d)** First, the revenue-maximizing rate is not the best rate: at 2/3, output would be 42 percent lower, and (c) shows that workers lose more in wages than the government collects (in this model, that is true at every positive rate). Second, capital and profits move between countries. Problem 3 showed that most countries cut their rates after 2000, and Problem 1 that firms can report their profits where taxes are low; a country with a rate far above its neighbors would lose part of its tax base.

</details>